In [1]:
!nvidia-smi

Sat Sep 19 20:10:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.2 MB/s eta 0:00:00


In [8]:
from ultralytics import YOLO
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [9]:
from google.colab import files

uploaded = files.upload()

Saving yolo_crack.zip to yolo_crack.zip


In [10]:
!unzip -q /content/yolo_crack.zip -d /content/

In [11]:
!ls /content/yolo_crack

data.yaml  images  labels


In [12]:
from pathlib import Path

base = Path("/content/yolo_crack")

for split in ["train", "val", "test"]:
    images = list((base / "images" / split).glob("*"))
    labels = list((base / "labels" / split).glob("*.txt"))

    print(f"{split.upper()}:")
    print(f"  Images: {len(images)}")
    print(f"  Labels: {len(labels)}")

TRAIN:
  Images: 2796
  Labels: 2796
VAL:
  Images: 697
  Labels: 697
TEST:
  Images: 1695
  Labels: 1695


In [13]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt")

print("Model ready:", type(model).__name__)

Model ready: YOLO


In [14]:
from pathlib import Path

yaml_path = Path("/content/yolo_crack/data.yaml")

yaml_path.write_text("""path: /content/yolo_crack
train: images/train
val: images/val
test: images/test

names:
  0: crack
""")

print(yaml_path.read_text())

path: /content/yolo_crack
train: images/train
val: images/val
test: images/test

names:
  0: crack



In [15]:
from pathlib import Path

base = Path("/content/yolo_crack")

print("Train:", (base / "images/train").exists())
print("Val:", (base / "images/val").exists())
print("Test:", (base / "images/test").exists())

Train: True
Val: True
Test: True


In [16]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/yolo_crack/data.yaml",
    epochs=50,
    imgsz=448,
    batch=16,
    device=0,
    workers=2,
    patience=15,
    project="/content/runs",
    name="crack_seg_baseline",
)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_crack/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=448, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=crack_seg_baseline, nbs=64, nms=None, opse

In [28]:
from ultralytics import YOLO
from pathlib import Path
import random

# Load the best trained model
model = YOLO("/content/runs/crack_seg_baseline/weights/best.pt")

# Get validation images
val_dir = Path("/content/yolo_crack/images/val")
val_images = list(val_dir.glob("*"))

# Select 8 random validation images
sample_images = random.sample(val_images, 8)

print("Selected validation images:")
for img in sample_images:
    print(img.name)

# Run predictions
results = model.predict(
    source=[str(img) for img in sample_images],
    conf=0.25,
    save=True,
    project="/content/runs",
    name="val_predictions",
    device=0,
)

print("\nPredictions completed successfully!")
print("Saved to: /content/runs/val_predictions")

Selected validation images:
CRACK500_20160308_073213_1_361.jpg
noncrack_noncrack_concrete_wall_25_5.jpg.jpg
CRACK500_20160308_160053_1_721.jpg
CRACK500_20160308_073245_1_1081.jpg
CRACK500_20160324_170542_1_361.jpg
CRACK500_20160328_150940_1_721.jpg
CRACK500_20160325_185829_1_1081.jpg
noncrack_noncrack_concrete_wall_76_32.jpg.jpg

0: 448x448 1 crack, 8.3ms
1: 448x448 (no detections), 8.3ms
2: 448x448 1 crack, 8.3ms
3: 448x448 1 crack, 8.3ms
4: 448x448 1 crack, 8.3ms
5: 448x448 (no detections), 8.3ms
6: 448x448 2 cracks, 8.3ms
7: 448x448 (no detections), 8.3ms
Speed: 1.5ms preprocess, 8.3ms inference, 4.3ms postprocess per image at shape (1, 3, 448, 448)
Results saved to /content/runs/val_predictions

Predictions completed successfully!
Saved to: /content/runs/val_predictions


In [29]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np

# Load the best trained model
model = YOLO("/content/runs/crack_seg_baseline/weights/best.pt")

val_images_dir = Path("/content/yolo_crack/images/val")
val_labels_dir = Path("/content/yolo_crack/labels/val")

ious = []
dices = []

def polygon_to_mask(label_path, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        values = list(map(float, line.split()))

        # Skip invalid labels
        if len(values) < 7:
            continue

        # First value = class
        coords = np.array(values[1:], dtype=np.float32)

        # Convert normalized coordinates to pixels
        points = coords.reshape(-1, 2)
        points[:, 0] *= width
        points[:, 1] *= height
        points = np.round(points).astype(np.int32)

        cv2.fillPoly(mask, [points], 1)

    return mask


for i, image_path in enumerate(sorted(val_images_dir.glob("*")), start=1):

    # Read image
    image = cv2.imread(str(image_path))

    if image is None:
        continue

    height, width = image.shape[:2]

    # Ground-truth mask
    label_path = val_labels_dir / f"{image_path.stem}.txt"
    gt_mask = polygon_to_mask(label_path, height, width)

    # Model prediction
    result = model.predict(
        source=str(image_path),
        conf=0.25,
        device=0,
        verbose=False
    )[0]

    # Create prediction mask
    pred_mask = np.zeros((height, width), dtype=np.uint8)

    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()

        for mask in masks:
            mask_resized = cv2.resize(
                mask,
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )
            pred_mask = np.maximum(
                pred_mask,
                (mask_resized > 0.5).astype(np.uint8)
            )

    # Convert to boolean
    gt = gt_mask.astype(bool)
    pred = pred_mask.astype(bool)

    # Intersection and union
    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    # IoU
    if union == 0:
        iou = 1.0 if not gt.any() and not pred.any() else 0.0
    else:
        iou = intersection / union

    # Dice
    denominator = gt.sum() + pred.sum()

    if denominator == 0:
        dice = 1.0
    else:
        dice = (2 * intersection) / denominator

    ious.append(iou)
    dices.append(dice)

    if i % 100 == 0:
        print(f"Processed {i}/{len(list(val_images_dir.glob('*')))} images")


print("\n" + "=" * 50)
print("VALIDATION IoU & DICE")
print("=" * 50)

print(f"Images evaluated: {len(ious)}")
print(f"Mean IoU:         {np.mean(ious):.4f}")
print(f"Mean Dice:        {np.mean(dices):.4f}")
print(f"Median IoU:       {np.median(ious):.4f}")
print(f"Median Dice:      {np.median(dices):.4f}")
print("=" * 50)

Processed 100/697 images
Processed 200/697 images
Processed 300/697 images
Processed 400/697 images
Processed 500/697 images
Processed 600/697 images

VALIDATION IoU & DICE
Images evaluated: 697
Mean IoU:         0.7101
Mean Dice:        0.7936
Median IoU:       0.7203
Median Dice:      0.8374


In [30]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# Load model
model = YOLO("/content/runs/crack_seg_baseline/weights/best.pt")

val_images_dir = Path("/content/yolo_crack/images/val")
val_labels_dir = Path("/content/yolo_crack/labels/val")

results_data = []


def polygon_to_mask(label_path, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        values = list(map(float, line.split()))

        if len(values) < 7:
            continue

        coords = np.array(values[1:], dtype=np.float32)
        points = coords.reshape(-1, 2)

        points[:, 0] *= width
        points[:, 1] *= height

        points = np.round(points).astype(np.int32)

        cv2.fillPoly(mask, [points], 1)

    return mask


image_paths = sorted(val_images_dir.glob("*"))

for i, image_path in enumerate(image_paths, start=1):

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    height, width = image.shape[:2]

    # -------------------------
    # Ground Truth
    # -------------------------
    label_path = val_labels_dir / f"{image_path.stem}.txt"

    gt_mask = polygon_to_mask(
        label_path,
        height,
        width
    )

    # -------------------------
    # Prediction
    # -------------------------
    result = model.predict(
        source=str(image_path),
        conf=0.25,
        device=0,
        verbose=False
    )[0]

    pred_mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    if result.masks is not None:

        masks = result.masks.data.cpu().numpy()

        for mask in masks:

            mask_resized = cv2.resize(
                mask,
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )

            pred_mask = np.maximum(
                pred_mask,
                (mask_resized > 0.5).astype(np.uint8)
            )

    # -------------------------
    # Metrics
    # -------------------------
    gt = gt_mask.astype(bool)
    pred = pred_mask.astype(bool)

    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    gt_pixels = gt.sum()
    pred_pixels = pred.sum()

    # IoU
    if union == 0:
        iou = 1.0 if gt_pixels == 0 and pred_pixels == 0 else 0.0
    else:
        iou = intersection / union

    # Dice
    denominator = gt_pixels + pred_pixels

    if denominator == 0:
        dice = 1.0
    else:
        dice = (2 * intersection) / denominator

    # -------------------------
    # Store results
    # -------------------------
    results_data.append({
        "image": image_path.name,
        "iou": iou,
        "dice": dice,
        "gt_pixels": gt_pixels,
        "pred_pixels": pred_pixels,
        "intersection": intersection,
        "union": union,
    })

    if i % 100 == 0:
        print(f"Processed {i}/{len(image_paths)}")


# Convert to DataFrame
df = pd.DataFrame(results_data)

# Sort from WORST to BEST
worst_10 = df.sort_values(
    by="iou",
    ascending=True
).head(10)

print("\n" + "=" * 70)
print("WORST 10 VALIDATION IMAGES BY IoU")
print("=" * 70)

print(
    worst_10[
        [
            "image",
            "iou",
            "dice",
            "gt_pixels",
            "pred_pixels"
        ]
    ].to_string(index=False)
)

# Save full results
output_path = "/content/validation_error_analysis.csv"
df.sort_values("iou").to_csv(
    output_path,
    index=False
)

print("\nFull results saved to:")
print(output_path)

Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

WORST 10 VALIDATION IMAGES BY IoU
                                 image  iou  dice  gt_pixels  pred_pixels
                           CFD_003.jpg  0.0   0.0       3152            0
      CRACK500_20160328_150932_1_1.jpg  0.0   0.0       3537            0
    CRACK500_20160328_150940_1_721.jpg  0.0   0.0        979            0
      CRACK500_20160307_144150_1_1.jpg  0.0   0.0       3035            0
   CRACK500_20160329_094055_1281_1.jpg  0.0   0.0       3142            0
  CRACK500_20160307_164517_641_361.jpg  0.0   0.0       2311            0
CRACK500_20160316_143603_1921_1081.jpg  0.0   0.0       7786            0
  CRACK500_20160318_181602_641_721.jpg  0.0   0.0       1086            0
   CRACK500_20160308_155720_1_1081.jpg  0.0   0.0       2150            0
CRACK500_20160316_144241_1921_1081.jpg  0.0   0.0       5450            0

Full results saved to:
/content/validation

In [31]:
# Classify validation images by prediction behavior

total = len(df)

missed_cracks = df[
    (df["gt_pixels"] > 0) &
    (df["pred_pixels"] == 0)
]

false_positives = df[
    (df["gt_pixels"] == 0) &
    (df["pred_pixels"] > 0)
]

perfect_empty = df[
    (df["gt_pixels"] == 0) &
    (df["pred_pixels"] == 0)
]

detected_cracks = df[
    (df["gt_pixels"] > 0) &
    (df["pred_pixels"] > 0)
]

print("=" * 60)
print("VALIDATION ERROR BREAKDOWN")
print("=" * 60)

print(f"Total images:                 {total}")

print(
    f"Missed cracks (FN):           "
    f"{len(missed_cracks)} "
    f"({len(missed_cracks)/total*100:.2f}%)"
)

print(
    f"False positives:              "
    f"{len(false_positives)} "
    f"({len(false_positives)/total*100:.2f}%)"
)

print(
    f"Correct empty predictions:    "
    f"{len(perfect_empty)} "
    f"({len(perfect_empty)/total*100:.2f}%)"
)

print(
    f"Cracks with prediction:       "
    f"{len(detected_cracks)} "
    f"({len(detected_cracks)/total*100:.2f}%)"
)

print("=" * 60)

VALIDATION ERROR BREAKDOWN
Total images:                 697
Missed cracks (FN):           31 (4.45%)
False positives:              0 (0.00%)
Correct empty predictions:    239 (34.29%)
Cracks with prediction:       427 (61.26%)


In [32]:
# Analyze segmentation quality on crack images only

crack_images = df[df["gt_pixels"] > 0].copy()

# Split by IoU quality
excellent = crack_images[crack_images["iou"] >= 0.75]

good = crack_images[
    (crack_images["iou"] >= 0.50) &
    (crack_images["iou"] < 0.75)
]

poor = crack_images[crack_images["iou"] < 0.50]

print("=" * 65)
print("CRACK SEGMENTATION QUALITY")
print("=" * 65)

print(f"Total crack images:       {len(crack_images)}")

print("\n--- Overall ---")
print(f"Mean IoU:                 {crack_images['iou'].mean():.4f}")
print(f"Median IoU:               {crack_images['iou'].median():.4f}")

print(f"\nMean Dice:                {crack_images['dice'].mean():.4f}")
print(f"Median Dice:             {crack_images['dice'].median():.4f}")

print("\n--- IoU Distribution ---")

print(
    f"IoU >= 0.75:             "
    f"{len(excellent)} "
    f"({len(excellent)/len(crack_images)*100:.2f}%)"
)

print(
    f"0.50 <= IoU < 0.75:      "
    f"{len(good)} "
    f"({len(good)/len(crack_images)*100:.2f}%)"
)

print(
    f"IoU < 0.50:              "
    f"{len(poor)} "
    f"({len(poor)/len(crack_images)*100:.2f}%)"
)

print("=" * 65)

CRACK SEGMENTATION QUALITY
Total crack images:       458

--- Overall ---
Mean IoU:                 0.5588
Median IoU:               0.6010

Mean Dice:                0.6859
Median Dice:             0.7508

--- IoU Distribution ---
IoU >= 0.75:             79 (17.25%)
0.50 <= IoU < 0.75:      245 (53.49%)
IoU < 0.50:              134 (29.26%)


In [33]:
# Analyze whether crack size affects segmentation quality

crack_images = df[df["gt_pixels"] > 0].copy()

# Calculate crack area as percentage of the image
image_area = 448 * 448

crack_images["gt_area_percent"] = (
    crack_images["gt_pixels"] / image_area * 100
)

# Split images by crack size
small_cracks = crack_images[
    crack_images["gt_area_percent"] < 1
]

medium_cracks = crack_images[
    (crack_images["gt_area_percent"] >= 1) &
    (crack_images["gt_area_percent"] < 5)
]

large_cracks = crack_images[
    crack_images["gt_area_percent"] >= 5
]

print("=" * 65)
print("CRACK SIZE vs SEGMENTATION QUALITY")
print("=" * 65)

print(f"Total crack images: {len(crack_images)}")

print("\n--- Small cracks (<1% of image) ---")
print(f"Images:    {len(small_cracks)}")
print(f"Mean IoU:  {small_cracks['iou'].mean():.4f}")
print(f"Mean Dice: {small_cracks['dice'].mean():.4f}")

print("\n--- Medium cracks (1% - 5%) ---")
print(f"Images:    {len(medium_cracks)}")
print(f"Mean IoU:  {medium_cracks['iou'].mean():.4f}")
print(f"Mean Dice: {medium_cracks['dice'].mean():.4f}")

print("\n--- Large cracks (>=5%) ---")
print(f"Images:    {len(large_cracks)}")
print(f"Mean IoU:  {large_cracks['iou'].mean():.4f}")
print(f"Mean Dice: {large_cracks['dice'].mean():.4f}")

print("=" * 65)

CRACK SIZE vs SEGMENTATION QUALITY
Total crack images: 458

--- Small cracks (<1% of image) ---
Images:    36
Mean IoU:  0.3224
Mean Dice: 0.4258

--- Medium cracks (1% - 5%) ---
Images:    215
Mean IoU:  0.5107
Mean Dice: 0.6441

--- Large cracks (>=5%) ---
Images:    207
Mean IoU:  0.6500
Mean Dice: 0.7744


In [34]:
# Analyze crack size among poor segmentation cases (IoU < 0.50)

poor = df[
    (df["gt_pixels"] > 0) &
    (df["iou"] < 0.50)
].copy()

image_area = 448 * 448
poor["gt_area_percent"] = poor["gt_pixels"] / image_area * 100

small = poor[poor["gt_area_percent"] < 1]
medium = poor[
    (poor["gt_area_percent"] >= 1) &
    (poor["gt_area_percent"] < 5)
]
large = poor[poor["gt_area_percent"] >= 5]

print("=" * 65)
print("POOR SEGMENTATION (< 0.50 IoU) vs CRACK SIZE")
print("=" * 65)

print(f"Poor segmentation images: {len(poor)}")

print("\n--- Small cracks (<1%) ---")
print(f"Images: {len(small)} ({len(small)/len(poor)*100:.2f}%)")

print("\n--- Medium cracks (1%-5%) ---")
print(f"Images: {len(medium)} ({len(medium)/len(poor)*100:.2f}%)")

print("\n--- Large cracks (>=5%) ---")
print(f"Images: {len(large)} ({len(large)/len(poor)*100:.2f}%)")

print("=" * 65)

POOR SEGMENTATION (< 0.50 IoU) vs CRACK SIZE
Poor segmentation images: 134

--- Small cracks (<1%) ---
Images: 23 (17.16%)

--- Medium cracks (1%-5%) ---
Images: 81 (60.45%)

--- Large cracks (>=5%) ---
Images: 30 (22.39%)


In [35]:
# Failure rate within each crack-size group

crack_images = df[df["gt_pixels"] > 0].copy()

image_area = 448 * 448
crack_images["gt_area_percent"] = (
    crack_images["gt_pixels"] / image_area * 100
)

# Define crack-size groups
crack_images["size_group"] = "Large"

crack_images.loc[
    crack_images["gt_area_percent"] < 5,
    "size_group"
] = "Medium"

crack_images.loc[
    crack_images["gt_area_percent"] < 1,
    "size_group"
] = "Small"

# Define poor segmentation
crack_images["poor_segmentation"] = crack_images["iou"] < 0.50

print("=" * 70)
print("SEGMENTATION FAILURE RATE BY CRACK SIZE")
print("=" * 70)

for group in ["Small", "Medium", "Large"]:
    subset = crack_images[crack_images["size_group"] == group]

    total = len(subset)
    poor = subset["poor_segmentation"].sum()
    rate = poor / total * 100

    print(f"\n--- {group} cracks ---")
    print(f"Total images: {total}")
    print(f"Poor segmentation (<0.50 IoU): {poor}")
    print(f"Failure rate: {rate:.2f}%")

print("\n" + "=" * 70)

SEGMENTATION FAILURE RATE BY CRACK SIZE

--- Small cracks ---
Total images: 36
Poor segmentation (<0.50 IoU): 23
Failure rate: 63.89%

--- Medium cracks ---
Total images: 215
Poor segmentation (<0.50 IoU): 81
Failure rate: 37.67%

--- Large cracks ---
Total images: 207
Poor segmentation (<0.50 IoU): 30
Failure rate: 14.49%



In [36]:
# Inspect failed small-crack cases visually

import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Small cracks that the model segmented poorly
small_failed = df[
    (df["gt_pixels"] > 0) &
    (df["gt_pixels"] / (448 * 448) * 100 < 1) &
    (df["iou"] < 0.50)
].copy()

# Sort from worst IoU to better IoU
small_failed = small_failed.sort_values("iou")

print("=" * 70)
print("FAILED SMALL-CRACK CASES")
print("=" * 70)
print(f"Total failed small-crack images: {len(small_failed)}")

# Show first 12 worst cases
samples = small_failed.head(12)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for ax, (_, row) in zip(axes, samples.iterrows()):

    image_name = row["image"]
    image_path = f"/content/yolo_crack/images/val/{image_name}"

    image = cv2.imread(image_path)

    if image is None:
        ax.set_title(f"Image not found\n{image_name}", fontsize=8)
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    ax.imshow(image)
    ax.set_title(
        f"IoU: {row['iou']:.3f} | "
        f"GT area: {row['gt_pixels']/(448*448)*100:.2f}%",
        fontsize=9
    )
    ax.axis("off")

plt.suptitle(
    "Worst Small-Crack Segmentation Cases",
    fontsize=16
)

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [37]:
# Analyze very-small vs small cracks

small_cracks = df[
    (df["gt_pixels"] > 0) &
    (df["gt_pixels"] / (448 * 448) * 100 < 1)
].copy()

small_cracks["gt_area_percent"] = (
    small_cracks["gt_pixels"] / (448 * 448) * 100
)

# Split small cracks into two groups
very_small = small_cracks[
    small_cracks["gt_area_percent"] < 0.5
]

small = small_cracks[
    (small_cracks["gt_area_percent"] >= 0.5) &
    (small_cracks["gt_area_percent"] < 1)
]

print("=" * 70)
print("SMALL CRACK ANALYSIS")
print("=" * 70)

for name, group in [
    ("Very small (<0.5%)", very_small),
    ("Small (0.5%-1%)", small)
]:

    poor = (group["iou"] < 0.50).sum()
    failure_rate = poor / len(group) * 100

    print(f"\n--- {name} ---")
    print(f"Images: {len(group)}")
    print(f"Mean IoU: {group['iou'].mean():.4f}")
    print(f"Median IoU: {group['iou'].median():.4f}")
    print(f"Poor segmentation (<0.50): {poor}")
    print(f"Failure rate: {failure_rate:.2f}%")

print("\n" + "=" * 70)

SMALL CRACK ANALYSIS

--- Very small (<0.5%) ---
Images: 8
Mean IoU: 0.0768
Median IoU: 0.0000
Poor segmentation (<0.50): 7
Failure rate: 87.50%

--- Small (0.5%-1%) ---
Images: 28
Mean IoU: 0.3926
Median IoU: 0.4359
Poor segmentation (<0.50): 16
Failure rate: 57.14%



In [38]:
# Analyze actual crack pixel size for very-small cracks

small_cracks = df[
    (df["gt_pixels"] > 0) &
    (df["gt_pixels"] / (448 * 448) * 100 < 1)
].copy()

small_cracks["gt_area_percent"] = (
    small_cracks["gt_pixels"] / (448 * 448) * 100
)

very_small = small_cracks[
    small_cracks["gt_area_percent"] < 0.5
].copy()

print("=" * 70)
print("VERY-SMALL CRACK PIXEL ANALYSIS")
print("=" * 70)

print(f"Images: {len(very_small)}")

print("\n--- Ground-truth crack pixels ---")
print(f"Minimum: {very_small['gt_pixels'].min():.0f} pixels")
print(f"Maximum: {very_small['gt_pixels'].max():.0f} pixels")
print(f"Mean:    {very_small['gt_pixels'].mean():.1f} pixels")
print(f"Median:  {very_small['gt_pixels'].median():.1f} pixels")

print("\n--- Ground-truth area percentage ---")
print(f"Minimum: {very_small['gt_area_percent'].min():.3f}%")
print(f"Maximum: {very_small['gt_area_percent'].max():.3f}%")
print(f"Mean:    {very_small['gt_area_percent'].mean():.3f}%")
print(f"Median:  {very_small['gt_area_percent'].median():.3f}%")

print("\n--- Pixel count per image ---")

for _, row in very_small.sort_values("gt_pixels").iterrows():
    print(
        f"{row['image']}: "
        f"{row['gt_pixels']:.0f} pixels "
        f"({row['gt_area_percent']:.3f}%), "
        f"IoU={row['iou']:.3f}"
    )

print("=" * 70)

VERY-SMALL CRACK PIXEL ANALYSIS
Images: 8

--- Ground-truth crack pixels ---
Minimum: 876 pixels
Maximum: 979 pixels
Mean:    930.9 pixels
Median:  928.0 pixels

--- Ground-truth area percentage ---
Minimum: 0.436%
Maximum: 0.488%
Mean:    0.464%
Median:  0.462%

--- Pixel count per image ---
CRACK500_20160222_164141_641_1081.jpg: 876 pixels (0.436%), IoU=0.000
CRACK500_20160328_154454_361_1.jpg: 886 pixels (0.441%), IoU=0.000
CRACK500_20160326_141808_641_1.jpg: 911 pixels (0.454%), IoU=0.000
CRACK500_20160316_143547_641_721.jpg: 917 pixels (0.457%), IoU=0.000
CRACK500_20160326_150337_1281_1.jpg: 939 pixels (0.468%), IoU=0.000
CRACK500_20160222_114759_1_361.jpg: 968 pixels (0.482%), IoU=0.510
CRACK500_20160328_151808_1921_1.jpg: 971 pixels (0.484%), IoU=0.104
CRACK500_20160328_150940_1_721.jpg: 979 pixels (0.488%), IoU=0.000


In [3]:
# Analyze image contrast for very-small crack cases

import cv2
import numpy as np

very_small_names = [
    "CRACK500_20160222_164141_641_1081.jpg",
    "CRACK500_20160328_154454_361_1.jpg",
    "CRACK500_20160326_141808_641_1.jpg",
    "CRACK500_20160316_143547_641_721.jpg",
    "CRACK500_20160326_150337_1281_1.jpg",
    "CRACK500_20160222_114759_1_361.jpg",
    "CRACK500_20160328_151808_1921_1.jpg",
    "CRACK500_20160328_150940_1_721.jpg",
]

print("=" * 70)
print("VERY-SMALL CRACK IMAGE CONTRAST ANALYSIS")
print("=" * 70)

for image_name in very_small_names:

    image_path = f"/content/yolo_crack/images/val/{image_name}"

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print(f"\nCould not read: {image_name}")
        continue

    # Basic global statistics
    mean = image.mean()
    std = image.std()
    min_val = image.min()
    max_val = image.max()

    print(f"\n{image_name}")
    print(f"  Mean intensity: {mean:.2f}")
    print(f"  Std intensity:  {std:.2f}")
    print(f"  Min intensity:  {min_val}")
    print(f"  Max intensity:  {max_val}")

print("\n" + "=" * 70)

VERY-SMALL CRACK IMAGE CONTRAST ANALYSIS

Could not read: CRACK500_20160222_164141_641_1081.jpg

Could not read: CRACK500_20160328_154454_361_1.jpg

Could not read: CRACK500_20160326_141808_641_1.jpg

Could not read: CRACK500_20160316_143547_641_721.jpg

Could not read: CRACK500_20160326_150337_1281_1.jpg

Could not read: CRACK500_20160222_114759_1_361.jpg

Could not read: CRACK500_20160328_151808_1921_1.jpg

Could not read: CRACK500_20160328_150940_1_721.jpg



In [4]:
# Local contrast analysis around the ground-truth crack

import cv2
import numpy as np

very_small_names = [
    "CRACK500_20160222_164141_641_1081.jpg",
    "CRACK500_20160328_154454_361_1.jpg",
    "CRACK500_20160326_141808_641_1.jpg",
    "CRACK500_20160316_143547_641_721.jpg",
    "CRACK500_20160222_114759_1_361.jpg",
    "CRACK500_20160328_151808_1921_1.jpg",
    "CRACK500_20160328_150940_1_721.jpg",
    "CRACK500_20160326_150337_1281_1.jpg",
]

print("=" * 90)
print("LOCAL CONTRAST ANALYSIS")
print("=" * 90)

for image_name in very_small_names:

    image_path = f"/content/yolo_crack/images/val/{image_name}"
    label_path = f"/content/yolo_crack/labels/val/{image_name.rsplit('.', 1)[0]}.txt"

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print(f"\nCould not read image: {image_name}")
        continue

    h, w = image.shape

    # Create ground-truth mask
    gt_mask = np.zeros((h, w), dtype=np.uint8)

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = list(map(float, line.strip().split()))

        if len(values) < 7:
            continue

        # Skip class index
        coords = values[1:]

        points = []

        for i in range(0, len(coords), 2):
            x = int(coords[i] * w)
            y = int(coords[i + 1] * h)
            points.append([x, y])

        points = np.array(points, dtype=np.int32)

        cv2.fillPoly(gt_mask, [points], 1)

    # Dilate GT mask to create local surrounding region
    kernel = np.ones((9, 9), np.uint8)
    dilated = cv2.dilate(gt_mask, kernel, iterations=1)

    # Ring = surrounding area excluding the crack itself
    ring = ((dilated == 1) & (gt_mask == 0))

    # Pixel intensities
    crack_pixels = image[gt_mask == 1]
    ring_pixels = image[ring]

    if len(crack_pixels) == 0 or len(ring_pixels) == 0:
        print(f"\n{image_name}")
        print("  Could not calculate local contrast.")
        continue

    crack_mean = crack_pixels.mean()
    ring_mean = ring_pixels.mean()

    crack_std = crack_pixels.std()
    ring_std = ring_pixels.std()

    absolute_contrast = abs(crack_mean - ring_mean)

    standardized_contrast = (
        absolute_contrast / (ring_std + 1e-8)
    )

    print(f"\n{image_name}")
    print(f"  Crack mean intensity:       {crack_mean:.2f}")
    print(f"  Surrounding mean intensity: {ring_mean:.2f}")
    print(f"  Absolute local contrast:    {absolute_contrast:.2f}")
    print(f"  Crack std:                  {crack_std:.2f}")
    print(f"  Surrounding std:            {ring_std:.2f}")
    print(f"  Standardized contrast:      {standardized_contrast:.3f}")

print("\n" + "=" * 90)

LOCAL CONTRAST ANALYSIS

Could not read image: CRACK500_20160222_164141_641_1081.jpg

Could not read image: CRACK500_20160328_154454_361_1.jpg

Could not read image: CRACK500_20160326_141808_641_1.jpg

Could not read image: CRACK500_20160316_143547_641_721.jpg

Could not read image: CRACK500_20160222_114759_1_361.jpg

Could not read image: CRACK500_20160328_151808_1921_1.jpg

Could not read image: CRACK500_20160328_150940_1_721.jpg

Could not read image: CRACK500_20160326_150337_1281_1.jpg



In [21]:
from ultralytics import YOLO

print("=" * 70)
print("EXPERIMENT 1 — SMALL-CRACK FOCUSED AUGMENTATION")
print("=" * 70)

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/yolo_crack/data.yaml",

    # Same core training setup as baseline
    epochs=50,
    imgsz=448,
    batch=16,
    device=0,
    workers=2,
    patience=15,

    # Experiment-specific augmentation
    degrees=5.0,
    translate=0.10,
    scale=0.30,

    fliplr=0.5,
    flipud=0.1,

    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.15,
    close_mosaic=10,

    # Save separately
    project="/content/runs",
    name="crack_seg_exp1_aug",

    # Reproducibility
    seed=42,
)

print("\n" + "=" * 70)
print("EXPERIMENT 1 TRAINING FINISHED")
print("=" * 70)

EXPERIMENT 1 — SMALL-CRACK FOCUSED AUGMENTATION
New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_crack/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=448, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4,

In [22]:
# ============================================================
# EXPERIMENT 1 — CUSTOM VALIDATION EVALUATION
# ============================================================

from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MODEL_PATH = "/content/runs/crack_seg_exp1_aug/weights/best.pt"
VAL_IMAGES = Path("/content/yolo_crack/images/val")
VAL_LABELS = Path("/content/yolo_crack/labels/val")

IMG_SIZE = 448
CONF_THRESHOLD = 0.25
MASK_THRESHOLD = 0.5

# ------------------------------------------------------------
# 2. Load Experiment 1 model
# ------------------------------------------------------------

model = YOLO(MODEL_PATH)

print("=" * 70)
print("EXPERIMENT 1 — CUSTOM VALIDATION EVALUATION")
print("=" * 70)
print(f"Model: {MODEL_PATH}")
print(f"Validation images: {VAL_IMAGES}")
print(f"Confidence threshold: {CONF_THRESHOLD}")
print("=" * 70)


# ------------------------------------------------------------
# 3. Helper: Convert YOLO polygon labels → binary mask
# ------------------------------------------------------------

def yolo_label_to_mask(label_path, width=448, height=448):
    """
    Convert YOLO segmentation polygons into one binary GT mask.
    Multiple crack objects are merged into a single mask.
    """

    mask = np.zeros((height, width), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        values = line.split()

        # class + at least 3 polygon points
        if len(values) < 7:
            continue

        # Skip class index
        coords = np.array(
            [float(x) for x in values[1:]],
            dtype=np.float32
        )

        # x,y pairs
        coords = coords.reshape(-1, 2)

        # Convert normalized coordinates → pixels
        coords[:, 0] *= width
        coords[:, 1] *= height

        polygon = np.round(coords).astype(np.int32)

        cv2.fillPoly(mask, [polygon], 1)

    return mask


# ------------------------------------------------------------
# 4. Evaluate every validation image
# ------------------------------------------------------------

image_paths = sorted(
    list(VAL_IMAGES.glob("*.jpg")) +
    list(VAL_IMAGES.glob("*.jpeg")) +
    list(VAL_IMAGES.glob("*.png"))
)

print(f"\nFound {len(image_paths)} validation images.")

results_data = []

for image_path in tqdm(image_paths, desc="Evaluating validation set"):

    # -------------------------
    # Ground Truth
    # -------------------------

    label_path = VAL_LABELS / f"{image_path.stem}.txt"

    gt_mask = yolo_label_to_mask(
        label_path,
        width=IMG_SIZE,
        height=IMG_SIZE
    )

    # -------------------------
    # Prediction
    # -------------------------

    prediction = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        verbose=False
    )[0]

    pred_mask = np.zeros(
        (IMG_SIZE, IMG_SIZE),
        dtype=np.uint8
    )

    if prediction.masks is not None:

        masks = prediction.masks.data.cpu().numpy()

        for instance_mask in masks:

            # Resize if necessary
            if instance_mask.shape != (IMG_SIZE, IMG_SIZE):
                instance_mask = cv2.resize(
                    instance_mask,
                    (IMG_SIZE, IMG_SIZE),
                    interpolation=cv2.INTER_NEAREST
                )

            binary_mask = (
                instance_mask > MASK_THRESHOLD
            ).astype(np.uint8)

            # Union all predicted crack instances
            pred_mask = np.maximum(
                pred_mask,
                binary_mask
            )

    # -------------------------
    # Calculate metrics
    # -------------------------

    gt_bool = gt_mask.astype(bool)
    pred_bool = pred_mask.astype(bool)

    intersection = np.logical_and(
        gt_bool,
        pred_bool
    ).sum()

    union = np.logical_or(
        gt_bool,
        pred_bool
    ).sum()

    gt_pixels = gt_bool.sum()
    pred_pixels = pred_bool.sum()

    # IoU
    if union == 0:
        iou = 1.0
    else:
        iou = intersection / union

    # Dice
    denominator = gt_pixels + pred_pixels

    if denominator == 0:
        dice = 1.0
    else:
        dice = (
            2 * intersection
            / denominator
        )

    # -------------------------
    # Error categories
    # -------------------------

    missed_crack = (
        gt_pixels > 0 and
        pred_pixels == 0
    )

    false_positive = (
        gt_pixels == 0 and
        pred_pixels > 0
    )

    correct_empty = (
        gt_pixels == 0 and
        pred_pixels == 0
    )

    crack_detected = (
        gt_pixels > 0 and
        pred_pixels > 0
    )

    # -------------------------
    # Crack size
    # -------------------------

    gt_area_percent = (
        gt_pixels /
        (IMG_SIZE * IMG_SIZE)
    ) * 100

    if gt_pixels == 0:
        size_group = "No crack"
    elif gt_area_percent < 0.5:
        size_group = "Very small"
    elif gt_area_percent < 1:
        size_group = "Small"
    elif gt_area_percent < 5:
        size_group = "Medium"
    else:
        size_group = "Large"

    results_data.append({
        "image": image_path.name,
        "gt_pixels": int(gt_pixels),
        "pred_pixels": int(pred_pixels),
        "gt_area_percent": gt_area_percent,
        "intersection": int(intersection),
        "union": int(union),
        "iou": iou,
        "dice": dice,
        "missed_crack": missed_crack,
        "false_positive": false_positive,
        "correct_empty": correct_empty,
        "crack_detected": crack_detected,
        "size_group": size_group
    })


# ------------------------------------------------------------
# 5. Create DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(results_data)

# Save detailed results
csv_path = "/content/validation_error_analysis_exp1.csv"
df.to_csv(csv_path, index=False)

print("\nEvaluation finished.")
print(f"Saved detailed results to:")
print(csv_path)


# ------------------------------------------------------------
# 6. Overall metrics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL VALIDATION RESULTS")
print("=" * 70)

print(f"Validation images : {len(df)}")
print(f"Mean IoU          : {df['iou'].mean():.4f}")
print(f"Median IoU        : {df['iou'].median():.4f}")
print(f"Mean Dice         : {df['dice'].mean():.4f}")
print(f"Median Dice       : {df['dice'].median():.4f}")


# ------------------------------------------------------------
# 7. Crack-only metrics
# ------------------------------------------------------------

crack_df = df[df["gt_pixels"] > 0].copy()

print("\n" + "=" * 70)
print("CRACK-ONLY RESULTS")
print("=" * 70)

print(f"Crack images      : {len(crack_df)}")
print(f"Mean IoU          : {crack_df['iou'].mean():.4f}")
print(f"Median IoU        : {crack_df['iou'].median():.4f}")
print(f"Mean Dice         : {crack_df['dice'].mean():.4f}")
print(f"Median Dice       : {crack_df['dice'].median():.4f}")


# ------------------------------------------------------------
# 8. Error breakdown
# ------------------------------------------------------------

total = len(df)

missed = int(df["missed_crack"].sum())
false_pos = int(df["false_positive"].sum())
correct_empty = int(df["correct_empty"].sum())
detected = int(df["crack_detected"].sum())

print("\n" + "=" * 70)
print("ERROR BREAKDOWN")
print("=" * 70)

print(f"Missed cracks        : {missed} / {total} ({missed/total*100:.2f}%)")
print(f"False positives      : {false_pos} / {total} ({false_pos/total*100:.2f}%)")
print(f"Correct empty        : {correct_empty} / {total} ({correct_empty/total*100:.2f}%)")
print(f"Cracks with predict. : {detected} / {total} ({detected/total*100:.2f}%)")

print(
    f"\nMissed among crack images: "
    f"{missed}/{len(crack_df)} "
    f"({missed/len(crack_df)*100:.2f}%)"
)


# ------------------------------------------------------------
# 9. IoU distribution — crack images
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CRACK IoU DISTRIBUTION")
print("=" * 70)

iou_075 = (crack_df["iou"] >= 0.75).sum()
iou_050_075 = (
    (crack_df["iou"] >= 0.50) &
    (crack_df["iou"] < 0.75)
).sum()
iou_below_050 = (crack_df["iou"] < 0.50).sum()

n_cracks = len(crack_df)

print(
    f"IoU >= 0.75     : {iou_075} "
    f"({iou_075/n_cracks*100:.2f}%)"
)

print(
    f"0.50 <= IoU < .75: {iou_050_075} "
    f"({iou_050_075/n_cracks*100:.2f}%)"
)

print(
    f"IoU < 0.50      : {iou_below_050} "
    f"({iou_below_050/n_cracks*100:.2f}%)"
)


# ------------------------------------------------------------
# 10. Crack size analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CRACK SIZE ANALYSIS")
print("=" * 70)

size_order = [
    "Very small",
    "Small",
    "Medium",
    "Large"
]

for group in size_order:

    group_df = df[df["size_group"] == group]

    if len(group_df) == 0:
        continue

    poor = (group_df["iou"] < 0.50).sum()

    print(f"\n{group}")
    print("-" * 30)
    print(f"Images       : {len(group_df)}")
    print(f"Mean IoU     : {group_df['iou'].mean():.4f}")
    print(f"Mean Dice    : {group_df['dice'].mean():.4f}")
    print(f"IoU < 0.50   : {poor}/{len(group_df)} "
          f"({poor/len(group_df)*100:.2f}%)")


# ------------------------------------------------------------
# 11. Worst 10 crack cases
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WORST 10 CRACK CASES")
print("=" * 70)

worst_cracks = (
    crack_df
    .sort_values("iou")
    .head(10)
)

print(
    worst_cracks[
        [
            "image",
            "gt_pixels",
            "pred_pixels",
            "gt_area_percent",
            "iou",
            "dice",
            "size_group",
            "missed_crack"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("DONE ✅")
print("=" * 70)

EXPERIMENT 1 — CUSTOM VALIDATION EVALUATION
Model: /content/runs/crack_seg_exp1_aug/weights/best.pt
Validation images: /content/yolo_crack/images/val
Confidence threshold: 0.25

Found 697 validation images.


Evaluating validation set: 100%|██████████| 697/697 [00:11<00:00, 61.40it/s]


Evaluation finished.
Saved detailed results to:
/content/validation_error_analysis_exp1.csv

OVERALL VALIDATION RESULTS
Validation images : 697
Mean IoU          : 0.7145
Median IoU        : 0.7170
Mean Dice         : 0.8009
Median Dice       : 0.8352

CRACK-ONLY RESULTS
Crack images      : 458
Mean IoU          : 0.5677
Median IoU        : 0.6017
Mean Dice         : 0.6992
Median Dice       : 0.7513

ERROR BREAKDOWN
Missed cracks        : 18 / 697 (2.58%)
False positives      : 1 / 697 (0.14%)
Correct empty        : 238 / 697 (34.15%)
Cracks with predict. : 440 / 697 (63.13%)

Missed among crack images: 18/458 (3.93%)

CRACK IoU DISTRIBUTION
IoU >= 0.75     : 84 (18.34%)
0.50 <= IoU < .75: 241 (52.62%)
IoU < 0.50      : 133 (29.04%)

CRACK SIZE ANALYSIS

Very small
------------------------------
Images       : 8
Mean IoU     : 0.2523
Mean Dice    : 0.3281
IoU < 0.50   : 5/8 (62.50%)

Small
------------------------------
Images       : 28
Mean IoU     : 0.4624
Mean Dice    : 0.6103
Io

In [23]:
import pandas as pd

baseline = pd.read_csv("/content/validation_error_analysis.csv")
exp1 = pd.read_csv("/content/validation_error_analysis_exp1.csv")

print("BASELINE COLUMNS:")
print(baseline.columns.tolist())

print("\nEXPERIMENT 1 COLUMNS:")
print(exp1.columns.tolist())

print("\nBASELINE SAMPLE:")
display(baseline.head())

print("\nEXPERIMENT 1 SAMPLE:")
display(exp1.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/validation_error_analysis.csv'

In [24]:
import pandas as pd

IMG_SIZE = 448
TOTAL_PIXELS = IMG_SIZE * IMG_SIZE

baseline = pd.read_csv("/content/validation_error_analysis.csv")
exp1 = pd.read_csv("/content/validation_error_analysis_exp1.csv")

def prepare_analysis(df, name):
    df = df.copy()

    # Calculate GT crack area percentage
    df["gt_area_percent"] = (
        df["gt_pixels"] / TOTAL_PIXELS * 100
    )

    # Same size groups for BOTH experiments
    def get_size_group(area):
        if area < 0.5:
            return "Very Small (<0.5%)"
        elif area < 1.0:
            return "Small (0.5–1%)"
        elif area < 5.0:
            return "Medium (1–5%)"
        else:
            return "Large (>=5%)"

    df["size_group"] = df["gt_area_percent"].apply(get_size_group)

    # Crack images only
    crack = df[df["gt_pixels"] > 0].copy()

    summary = crack.groupby("size_group").agg(
        images=("iou", "count"),
        mean_iou=("iou", "mean"),
        median_iou=("iou", "median"),
        mean_dice=("dice", "mean"),
        poor_iou=("iou", lambda x: (x < 0.5).sum()),
        missed=("pred_pixels", lambda x: (x == 0).sum())
    ).reset_index()

    summary["failure_rate_%"] = (
        summary["poor_iou"] / summary["images"] * 100
    )

    summary["miss_rate_%"] = (
        summary["missed"] / summary["images"] * 100
    )

    # False positives on non-crack images
    noncrack = df[df["gt_pixels"] == 0]
    false_positives = (noncrack["pred_pixels"] > 0).sum()

    print("\n" + "="*70)
    print(name)
    print("="*70)

    print(f"Total images: {len(df)}")
    print(f"Crack images: {len(crack)}")
    print(f"Non-crack images: {len(noncrack)}")
    print(f"False positives: {false_positives}")

    print("\nSize-group analysis:")
    display(summary)

    return df, summary


baseline_prepared, baseline_summary = prepare_analysis(
    baseline,
    "BASELINE"
)

exp1_prepared, exp1_summary = prepare_analysis(
    exp1,
    "EXPERIMENT 1"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/validation_error_analysis.csv'

In [25]:
from ultralytics import YOLO

# Fresh pretrained model
model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/yolo_crack/data.yaml",

    # Same core setup as Baseline + Experiment 1
    epochs=50,
    imgsz=448,
    batch=16,
    device=0,
    workers=2,
    patience=15,

    # Same augmentation settings as Experiment 1
    degrees=5.0,
    translate=0.10,
    scale=0.30,
    fliplr=0.5,
    flipud=0.1,

    # Experiment 2 change
    mosaic=0.0,

    mixup=0.0,
    copy_paste=0.15,
    close_mosaic=10,

    seed=42,

    project="/content/runs",
    name="crack_seg_exp2_no_mosaic",
)

New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_crack/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=448, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo1

In [26]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from tqdm.auto import tqdm

# ============================================================
# EXPERIMENT 2 — FULL CUSTOM VALIDATION
# ============================================================

MODEL_PATH = "/content/runs/crack_seg_exp2_no_mosaic/weights/best.pt"

VAL_IMAGES = "/content/yolo_crack/images/val"
VAL_LABELS = "/content/yolo_crack/labels/val"

IMG_SIZE = 448
CONF_THRESHOLD = 0.25
MASK_THRESHOLD = 0.5

TOTAL_PIXELS = IMG_SIZE * IMG_SIZE

print("Loading model...")
model = YOLO(MODEL_PATH)

# ------------------------------------------------------------
# Helper: rasterize YOLO segmentation polygons
# ------------------------------------------------------------

def yolo_label_to_mask(label_path, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)

    if not os.path.exists(label_path):
        return mask

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 7:
            continue

        # First value = class
        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        )

        # x1,y1,x2,y2,...
        coords = coords.reshape(-1, 2)

        # Convert normalized coordinates to pixels
        points = np.zeros_like(coords)
        points[:, 0] = coords[:, 0] * width
        points[:, 1] = coords[:, 1] * height

        points = np.round(points).astype(np.int32)

        # Keep points inside image
        points[:, 0] = np.clip(points[:, 0], 0, width - 1)
        points[:, 1] = np.clip(points[:, 1], 0, height - 1)

        cv2.fillPoly(mask, [points], 1)

    return mask


# ------------------------------------------------------------
# Get validation images
# ------------------------------------------------------------

image_files = sorted([
    f for f in os.listdir(VAL_IMAGES)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Validation images: {len(image_files)}")


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

results_data = []

for image_name in tqdm(image_files, desc="Evaluating Experiment 2"):

    image_path = os.path.join(VAL_IMAGES, image_name)

    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(VAL_LABELS, label_name)

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        print(f"WARNING: Could not read {image_name}")
        continue

    height, width = image.shape[:2]

    # -------------------------
    # Ground Truth Mask
    # -------------------------

    gt_mask = yolo_label_to_mask(
        label_path,
        width,
        height
    )

    gt_pixels = int(gt_mask.sum())

    # -------------------------
    # Prediction
    # -------------------------

    prediction = model.predict(
        source=image_path,
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        verbose=False
    )[0]

    pred_mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    if prediction.masks is not None:

        masks = prediction.masks.data.cpu().numpy()

        for m in masks:

            # Resize predicted mask to original image size
            resized = cv2.resize(
                m,
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )

            binary = (
                resized >= MASK_THRESHOLD
            ).astype(np.uint8)

            # Union all predicted masks
            pred_mask = np.maximum(
                pred_mask,
                binary
            )

    pred_pixels = int(pred_mask.sum())

    # -------------------------
    # IoU / Dice
    # -------------------------

    intersection = int(
        np.logical_and(
            gt_mask,
            pred_mask
        ).sum()
    )

    union = int(
        np.logical_or(
            gt_mask,
            pred_mask
        ).sum()
    )

    if union > 0:
        iou = intersection / union
    else:
        # Both GT and prediction are empty
        iou = 1.0

    denominator = gt_pixels + pred_pixels

    if denominator > 0:
        dice = (
            2 * intersection / denominator
        )
    else:
        dice = 1.0

    # -------------------------
    # Area percentage
    # -------------------------

    gt_area_percent = (
        gt_pixels / (width * height) * 100
    )

    # -------------------------
    # Error categories
    # -------------------------

    missed_crack = (
        gt_pixels > 0 and pred_pixels == 0
    )

    false_positive = (
        gt_pixels == 0 and pred_pixels > 0
    )

    correct_empty = (
        gt_pixels == 0 and pred_pixels == 0
    )

    crack_detected = (
        gt_pixels > 0 and pred_pixels > 0
    )

    # -------------------------
    # Size group
    # -------------------------

    if gt_pixels == 0:
        size_group = "No Crack"
    elif gt_area_percent < 0.5:
        size_group = "Very Small (<0.5%)"
    elif gt_area_percent < 1.0:
        size_group = "Small (0.5–1%)"
    elif gt_area_percent < 5.0:
        size_group = "Medium (1–5%)"
    else:
        size_group = "Large (>=5%)"

    results_data.append({
        "image": image_name,
        "gt_pixels": gt_pixels,
        "pred_pixels": pred_pixels,
        "gt_area_percent": gt_area_percent,
        "intersection": intersection,
        "union": union,
        "iou": iou,
        "dice": dice,
        "missed_crack": missed_crack,
        "false_positive": false_positive,
        "correct_empty": correct_empty,
        "crack_detected": crack_detected,
        "size_group": size_group
    })


# ============================================================
# RESULTS DATAFRAME
# ============================================================

df = pd.DataFrame(results_data)

print("\nEvaluation complete.")
print(f"Images evaluated: {len(df)}")


# ============================================================
# 1. OVERALL METRICS
# ============================================================

overall_mean_iou = df["iou"].mean()
overall_median_iou = df["iou"].median()

overall_mean_dice = df["dice"].mean()
overall_median_dice = df["dice"].median()

print("\n" + "=" * 70)
print("1. OVERALL METRICS")
print("=" * 70)

print(f"Mean IoU:       {overall_mean_iou:.4f}")
print(f"Median IoU:     {overall_median_iou:.4f}")
print(f"Mean Dice:      {overall_mean_dice:.4f}")
print(f"Median Dice:    {overall_median_dice:.4f}")


# ============================================================
# 2. CRACK-ONLY METRICS
# ============================================================

crack_df = df[df["gt_pixels"] > 0].copy()
noncrack_df = df[df["gt_pixels"] == 0].copy()

crack_mean_iou = crack_df["iou"].mean()
crack_median_iou = crack_df["iou"].median()

crack_mean_dice = crack_df["dice"].mean()
crack_median_dice = crack_df["dice"].median()

print("\n" + "=" * 70)
print("2. CRACK-ONLY METRICS")
print("=" * 70)

print(f"Crack images:   {len(crack_df)}")

print(f"Mean IoU:       {crack_mean_iou:.4f}")
print(f"Median IoU:     {crack_median_iou:.4f}")
print(f"Mean Dice:      {crack_mean_dice:.4f}")
print(f"Median Dice:    {crack_median_dice:.4f}")


# ============================================================
# 3. ERROR BREAKDOWN
# ============================================================

missed = int(df["missed_crack"].sum())
false_positives = int(df["false_positive"].sum())
correct_empty = int(df["correct_empty"].sum())
crack_detected = int(df["crack_detected"].sum())

miss_rate = (
    missed / len(crack_df) * 100
)

false_positive_rate = (
    false_positives / len(noncrack_df) * 100
)

print("\n" + "=" * 70)
print("3. ERROR BREAKDOWN")
print("=" * 70)

print(f"Total images:          {len(df)}")
print(f"Crack images:          {len(crack_df)}")
print(f"Non-crack images:      {len(noncrack_df)}")

print(f"\nMissed cracks:          {missed}")
print(f"Miss rate:              {miss_rate:.2f}%")

print(f"\nFalse positives:        {false_positives}")
print(f"False positive rate:    {false_positive_rate:.2f}%")

print(f"\nCorrect empty:          {correct_empty}")
print(f"Cracks detected:        {crack_detected}")


# ============================================================
# 4. IoU DISTRIBUTION — CRACK IMAGES ONLY
# ============================================================

iou_ge_075 = int((crack_df["iou"] >= 0.75).sum())
iou_050_075 = int(
    (
        (crack_df["iou"] >= 0.50) &
        (crack_df["iou"] < 0.75)
    ).sum()
)

iou_lt_050 = int(
    (crack_df["iou"] < 0.50).sum()
)

print("\n" + "=" * 70)
print("4. IoU DISTRIBUTION — CRACK IMAGES")
print("=" * 70)

print(
    f"IoU >= 0.75:       {iou_ge_075} "
    f"({iou_ge_075 / len(crack_df) * 100:.2f}%)"
)

print(
    f"0.50 <= IoU < 0.75: {iou_050_075} "
    f"({iou_050_075 / len(crack_df) * 100:.2f}%)"
)

print(
    f"IoU < 0.50:        {iou_lt_050} "
    f"({iou_lt_050 / len(crack_df) * 100:.2f}%)"
)


# ============================================================
# 5. SIZE-GROUP ANALYSIS
# ============================================================

size_order = [
    "Very Small (<0.5%)",
    "Small (0.5–1%)",
    "Medium (1–5%)",
    "Large (>=5%)"
]

size_results = []

for group in size_order:

    group_df = crack_df[
        crack_df["size_group"] == group
    ]

    if len(group_df) == 0:
        continue

    poor_iou = int(
        (group_df["iou"] < 0.5).sum()
    )

    missed_group = int(
        group_df["missed_crack"].sum()
    )

    size_results.append({
        "size_group": group,
        "images": len(group_df),
        "mean_iou": group_df["iou"].mean(),
        "median_iou": group_df["iou"].median(),
        "mean_dice": group_df["dice"].mean(),
        "poor_iou": poor_iou,
        "missed": missed_group,
        "failure_rate_%": (
            poor_iou / len(group_df) * 100
        ),
        "miss_rate_%": (
            missed_group / len(group_df) * 100
        )
    })

size_summary = pd.DataFrame(size_results)

print("\n" + "=" * 70)
print("5. SIZE-GROUP ANALYSIS")
print("=" * 70)

display(size_summary)


# ============================================================
# 6. WORST 10 CRACK CASES
# ============================================================

worst_cracks = (
    crack_df
    .sort_values("iou", ascending=True)
    .head(10)
)

print("\n" + "=" * 70)
print("6. WORST 10 CRACK CASES")
print("=" * 70)

display(
    worst_cracks[
        [
            "image",
            "gt_pixels",
            "pred_pixels",
            "gt_area_percent",
            "iou",
            "dice",
            "missed_crack",
            "size_group"
        ]
    ]
)


# ============================================================
# 7. WORST FALSE POSITIVES
# ============================================================

worst_fp = (
    noncrack_df[
        noncrack_df["pred_pixels"] > 0
    ]
    .sort_values("pred_pixels", ascending=False)
)

print("\n" + "=" * 70)
print("7. FALSE POSITIVE CASES")
print("=" * 70)

if len(worst_fp) > 0:
    display(
        worst_fp[
            [
                "image",
                "gt_pixels",
                "pred_pixels",
                "iou",
                "dice"
            ]
        ]
    )
else:
    print("No false positives.")


# ============================================================
# 8. MISSED CRACK CASES
# ============================================================

missed_df = (
    crack_df[
        crack_df["missed_crack"]
    ]
    .sort_values("gt_pixels")
)

print("\n" + "=" * 70)
print("8. MISSED CRACKS")
print("=" * 70)

print(f"Missed cracks: {len(missed_df)}")

if len(missed_df) > 0:
    display(
        missed_df[
            [
                "image",
                "gt_pixels",
                "gt_area_percent",
                "size_group",
                "iou",
                "dice"
            ]
        ].head(20)
    )


# ============================================================
# 9. SAVE FULL CSV
# ============================================================

output_csv = "/content/validation_error_analysis_exp2.csv"

df.to_csv(
    output_csv,
    index=False
)

print("\n" + "=" * 70)
print("9. FILE SAVED")
print("=" * 70)

print(output_csv)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("10. EXPERIMENT 2 FINAL SUMMARY")
print("=" * 70)

print(f"""
Overall Mean IoU:          {overall_mean_iou:.4f}
Overall Median IoU:        {overall_median_iou:.4f}

Overall Mean Dice:         {overall_mean_dice:.4f}
Overall Median Dice:       {overall_median_dice:.4f}

Crack-only Mean IoU:       {crack_mean_iou:.4f}
Crack-only Median IoU:     {crack_median_iou:.4f}

Crack-only Mean Dice:      {crack_mean_dice:.4f}
Crack-only Median Dice:    {crack_median_dice:.4f}

Missed cracks:             {missed}
Miss rate:                 {miss_rate:.2f}%

False positives:           {false_positives}
False positive rate:       {false_positive_rate:.2f}%

IoU >= 0.75:               {iou_ge_075}
IoU 0.50–0.75:             {iou_050_075}
IoU < 0.50:                {iou_lt_050}
""")

Loading model...
Validation images: 697


Evaluating Experiment 2:   0%|          | 0/697 [00:00<?, ?it/s]


Evaluation complete.
Images evaluated: 697

1. OVERALL METRICS
Mean IoU:       0.7047
Median IoU:     0.7166
Mean Dice:      0.7885
Median Dice:    0.8349

2. CRACK-ONLY METRICS
Crack images:   458
Mean IoU:       0.5507
Median IoU:     0.5872
Mean Dice:      0.6782
Median Dice:    0.7399

3. ERROR BREAKDOWN
Total images:          697
Crack images:          458
Non-crack images:      239

Missed cracks:          34
Miss rate:              7.42%

False positives:        0
False positive rate:    0.00%

Correct empty:          239
Cracks detected:        424

4. IoU DISTRIBUTION — CRACK IMAGES
IoU >= 0.75:       71 (15.50%)
0.50 <= IoU < 0.75: 241 (52.62%)
IoU < 0.50:        146 (31.88%)

5. SIZE-GROUP ANALYSIS


,size_group,images,mean_iou,median_iou,mean_dice,poor_iou,missed,failure_rate_%,miss_rate_%
0,Very Small (<0.5%),8,0.000000,0.000000,0.000000,8,8,100.000000,100.000000
1,Small (0.5–1%),28,0.380482,0.495204,0.503365,15,6,53.571429,21.428571
2,Medium (1–5%),215,0.514823,0.540310,0.651739,87,14,40.465116,6.511628
3,Large (>=5%),207,0.632164,0.662299,0.755518,36,6,17.391304,2.898551



6. WORST 10 CRACK CASES


,image,gt_pixels,pred_pixels,gt_area_percent,iou,dice,missed_crack,size_group
445,CRACK500_20160329_094452_1281_1081.jpg,2816,0,1.403061,0.0,0.0,True,Medium (1–5%)
260,CRACK500_20160326_150337_1281_1.jpg,939,0,0.467853,0.0,0.0,True,Very Small (<0.5%)
267,CRACK500_20160326_162538_1281_721.jpg,11418,0,5.688975,0.0,0.0,True,Large (>=5%)
263,CRACK500_20160326_162529_641_1081.jpg,2370,0,1.180843,0.0,0.0,True,Medium (1–5%)
137,CRACK500_20160307_163641_1281_721.jpg,4329,0,2.156908,0.0,0.0,True,Medium (1–5%)
294,CRACK500_20160328_151151_1281_721.jpg,3902,0,1.944157,0.0,0.0,True,Medium (1–5%)
284,CRACK500_20160328_150940_1_721.jpg,979,0,0.487783,0.0,0.0,True,Very Small (<0.5%)
278,CRACK500_20160328_150932_1_1.jpg,3537,0,1.762297,0.0,0.0,True,Medium (1–5%)
123,CRACK500_20160307_144150_1_1.jpg,3035,0,1.512177,0.0,0.0,True,Medium (1–5%)
167,CRACK500_20160308_073356_641_721.jpg,2102,0,1.047313,0.0,0.0,True,Medium (1–5%)



7. FALSE POSITIVE CASES
No false positives.

8. MISSED CRACKS
Missed cracks: 34


,image,gt_pixels,gt_area_percent,size_group,iou,dice
66,CRACK500_20160222_164141_641_1081.jpg,876,0.436464,Very Small (<0.5%),0.0,0.0
428,CRACK500_20160328_154454_361_1.jpg,886,0.441446,Very Small (<0.5%),0.0,0.0
251,CRACK500_20160326_141808_641_1.jpg,911,0.453902,Very Small (<0.5%),0.0,0.0
211,CRACK500_20160316_143547_641_721.jpg,917,0.456892,Very Small (<0.5%),0.0,0.0
260,CRACK500_20160326_150337_1281_1.jpg,939,0.467853,Very Small (<0.5%),0.0,0.0
36,CRACK500_20160222_114759_1_361.jpg,968,0.482302,Very Small (<0.5%),0.0,0.0
326,CRACK500_20160328_151808_1921_1.jpg,971,0.483797,Very Small (<0.5%),0.0,0.0
284,CRACK500_20160328_150940_1_721.jpg,979,0.487783,Very Small (<0.5%),0.0,0.0
42,CRACK500_20160222_115305_1921_1081.jpg,1015,0.505720,Small (0.5–1%),0.0,0.0
28,CRACK500_20160222_081113_1281_361.jpg,1047,0.521664,Small (0.5–1%),0.0,0.0



9. FILE SAVED
/content/validation_error_analysis_exp2.csv

10. EXPERIMENT 2 FINAL SUMMARY

Overall Mean IoU:          0.7047
Overall Median IoU:        0.7166

Overall Mean Dice:         0.7885
Overall Median Dice:       0.8349

Crack-only Mean IoU:       0.5507
Crack-only Median IoU:     0.5872

Crack-only Mean Dice:      0.6782
Crack-only Median Dice:    0.7399

Missed cracks:             34
Miss rate:                 7.42%

False positives:           0
False positive rate:       0.00%

IoU >= 0.75:               71
IoU 0.50–0.75:             241
IoU < 0.50:                146



In [39]:
import os
import cv2
import numpy as np
import shutil
from pathlib import Path

# ============================================================
# EXP3 — Crack-Focused Cropping
# ============================================================

SOURCE_ROOT = Path("/content/yolo_crack")

EXP3_ROOT = Path("/content/yolo_crack_exp3")

# Clean previous Exp3 dataset if it exists
if EXP3_ROOT.exists():
    shutil.rmtree(EXP3_ROOT)

# Create structure
for split in ["train", "val", "test"]:
    (EXP3_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (EXP3_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Helper: copy original image + label
# ------------------------------------------------------------

def copy_sample(split, image_name):
    src_img = SOURCE_ROOT / "images" / split / image_name
    src_lbl = SOURCE_ROOT / "labels" / split / (Path(image_name).stem + ".txt")

    dst_img = EXP3_ROOT / "images" / split / image_name
    dst_lbl = EXP3_ROOT / "labels" / split / src_lbl.name

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)


# ------------------------------------------------------------
# Keep validation and test ORIGINAL
# ------------------------------------------------------------

for split in ["val", "test"]:
    for image_path in (SOURCE_ROOT / "images" / split).glob("*.jpg"):
        copy_sample(split, image_path.name)

print("Validation and test copied unchanged.")


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_images = SOURCE_ROOT / "images" / "train"
train_labels = SOURCE_ROOT / "labels" / "train"

original_count = 0
crop_count = 0

CROP_SIZE = 224
MARGIN = 32


for image_path in train_images.glob("*.jpg"):

    label_path = train_labels / f"{image_path.stem}.txt"

    # Copy original training image + label
    copy_sample("train", image_path.name)
    original_count += 1

    # Read label
    if not label_path.exists():
        continue

    lines = label_path.read_text().strip().splitlines()

    # Skip non-crack images
    if not lines:
        continue

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    h, w = image.shape[:2]

    # Collect all polygon points
    all_points = []

    for line in lines:
        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        # YOLO normalized -> pixel coordinates
        coords[:, 0] *= w
        coords[:, 1] *= h

        all_points.append(coords)

    if not all_points:
        continue

    all_points = np.vstack(all_points)

    # Bounding box around all crack pixels/polygon points
    x_min = np.min(all_points[:, 0])
    y_min = np.min(all_points[:, 1])
    x_max = np.max(all_points[:, 0])
    y_max = np.max(all_points[:, 1])

    # Center of crack region
    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2

    # --------------------------------------------------------
    # 224x224 crop centered around crack
    # --------------------------------------------------------

    half = CROP_SIZE // 2

    crop_x1 = int(round(cx - half))
    crop_y1 = int(round(cy - half))

    # Keep crop inside image
    crop_x1 = max(0, min(crop_x1, w - CROP_SIZE))
    crop_y1 = max(0, min(crop_y1, h - CROP_SIZE))

    crop_x2 = crop_x1 + CROP_SIZE
    crop_y2 = crop_y1 + CROP_SIZE

    crop = image[crop_y1:crop_y2, crop_x1:crop_x2]

    if crop.shape[0] != CROP_SIZE or crop.shape[1] != CROP_SIZE:
        continue

    # --------------------------------------------------------
    # Create cropped mask from original polygons
    # --------------------------------------------------------

    mask = np.zeros((h, w), dtype=np.uint8)

    for points in all_points.reshape(-1, 2):
        pass

    # Draw every original polygon
    for line in lines:
        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            [float(x) for x in parts[1:]],
            dtype=np.float32
        ).reshape(-1, 2)

        coords[:, 0] *= w
        coords[:, 1] *= h

        pts = np.round(coords).astype(np.int32)

        cv2.fillPoly(mask, [pts], 255)

    crop_mask = mask[crop_y1:crop_y2, crop_x1:crop_x2]

    # --------------------------------------------------------
    # Extract contours from cropped mask
    # This avoids invalid polygon clipping.
    # --------------------------------------------------------

    contours, _ = cv2.findContours(
        crop_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    new_labels = []

    for contour in contours:

        if len(contour) < 3:
            continue

        area = cv2.contourArea(contour)

        if area < 2:
            continue

        contour = contour.reshape(-1, 2).astype(np.float32)

        # Normalize to crop dimensions
        contour[:, 0] /= CROP_SIZE
        contour[:, 1] /= CROP_SIZE

        contour = np.clip(contour, 0.0, 1.0)

        coords = " ".join(
            f"{x:.6f} {y:.6f}"
            for x, y in contour
        )

        new_labels.append(f"0 {coords}")

    if not new_labels:
        continue

    # --------------------------------------------------------
    # Resize crop back to 448x448
    # --------------------------------------------------------

    crop_448 = cv2.resize(
        crop,
        (448, 448),
        interpolation=cv2.INTER_LINEAR
    )

    # Save
    crop_name = f"{image_path.stem}_crop.jpg"
    crop_label_name = f"{image_path.stem}_crop.txt"

    cv2.imwrite(
        str(EXP3_ROOT / "images" / "train" / crop_name),
        crop_448
    )

    # Labels remain normalized, so no coordinate change needed
    (EXP3_ROOT / "labels" / "train" / crop_label_name).write_text(
        "\n".join(new_labels)
    )

    crop_count += 1


# ------------------------------------------------------------
# Create data.yaml
# ------------------------------------------------------------

yaml_text = """path: /content/yolo_crack_exp3
train: images/train
val: images/val
test: images/test

names:
  0: crack
"""

(EXP3_ROOT / "data.yaml").write_text(yaml_text)

print("\n" + "=" * 60)
print("EXP3 DATASET CREATED")
print("=" * 60)
print(f"Original train images: {original_count}")
print(f"Additional crack crops: {crop_count}")
print(f"Total train images:     {original_count + crop_count}")
print(f"Validation images:      {len(list((EXP3_ROOT / 'images' / 'val').glob('*.jpg')))}")
print(f"Test images:            {len(list((EXP3_ROOT / 'images' / 'test').glob('*.jpg')))}")
print(f"Dataset: {EXP3_ROOT}")

Validation and test copied unchanged.

EXP3 DATASET CREATED
Original train images: 2796
Additional crack crops: 1797
Total train images:     4593
Validation images:      697
Test images:            1695
Dataset: /content/yolo_crack_exp3


In [40]:
from pathlib import Path

EXP3_ROOT = Path("/content/yolo_crack_exp3")

for split in ["train", "val", "test"]:
    image_dir = EXP3_ROOT / "images" / split
    label_dir = EXP3_ROOT / "labels" / split

    images = list(image_dir.glob("*.jpg"))
    labels = list(label_dir.glob("*.txt"))

    image_names = {x.stem for x in images}
    label_names = {x.stem for x in labels}

    missing_labels = image_names - label_names
    extra_labels = label_names - image_names

    invalid = 0
    empty = 0
    objects = 0

    for label_path in labels:
        lines = label_path.read_text().strip().splitlines()

        if not lines:
            empty += 1
            continue

        for line in lines:
            parts = line.split()

            if len(parts) < 7:
                invalid += 1
                continue

            try:
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]

                if cls != 0 or len(coords) % 2 != 0:
                    invalid += 1
                    continue

                if any(x < 0 or x > 1 for x in coords):
                    invalid += 1
                    continue

                objects += 1

            except:
                invalid += 1

    print("\n" + "=" * 55)
    print(split.upper())
    print("=" * 55)
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Missing labels:", len(missing_labels))
    print("Extra labels:", len(extra_labels))
    print("Empty labels:", empty)
    print("Objects:", objects)
    print("Invalid labels:", invalid)


TRAIN
Images: 4593
Labels: 4593
Missing labels: 0
Extra labels: 0
Empty labels: 960
Objects: 7046
Invalid labels: 0

VAL
Images: 697
Labels: 697
Missing labels: 0
Extra labels: 0
Empty labels: 239
Objects: 1094
Invalid labels: 0

TEST
Images: 1695
Labels: 1695
Missing labels: 0
Extra labels: 0
Empty labels: 221
Objects: 32088
Invalid labels: 0


In [41]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="/content/yolo_crack_exp3/data.yaml",
    epochs=50,
    imgsz=448,
    batch=16,
    device=0,
    workers=2,
    patience=15,

    project="/content/runs",
    name="crack_seg_exp3_crop",

    pretrained=True,

    # Same augmentation setup as Exp1
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.1,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.15,
    close_mosaic=10,

    seed=42
)

New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_crack_exp3/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=448, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import shutil
from pathlib import Path

SAVE_ROOT = Path("/content/drive/MyDrive/crack-segmentation-project")

SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Exp3 dataset
shutil.copytree(
    "/content/yolo_crack_exp3",
    SAVE_ROOT / "yolo_crack_exp3",
    dirs_exist_ok=True
)

# Exp3 training results + weights
shutil.copytree(
    "/content/runs/crack_seg_exp3_crop",
    SAVE_ROOT / "crack_seg_exp3_crop",
    dirs_exist_ok=True
)

print("✅ Exp3 files saved to Google Drive:")
print(SAVE_ROOT)

✅ Exp3 files saved to Google Drive:
/content/drive/MyDrive/crack-segmentation-project


In [44]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# =========================
# Exp3 - Custom Validation
# =========================

MODEL_PATH = "/content/runs/crack_seg_exp3_crop/weights/best.pt"
VAL_IMAGES = Path("/content/yolo_crack/images/val")
VAL_LABELS = Path("/content/yolo_crack/labels/val")

model = YOLO(MODEL_PATH)

# -------------------------
# Helpers
# -------------------------

def load_yolo_mask(label_path, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()
        if len(parts) < 7:
            continue

        coords = np.array(list(map(float, parts[1:])), dtype=np.float32)
        coords = coords.reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


def calculate_iou(gt, pred):
    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    if union == 0:
        return 1.0

    return intersection / union


def calculate_dice(gt, pred):
    gt_sum = gt.sum()
    pred_sum = pred.sum()

    if gt_sum + pred_sum == 0:
        return 1.0

    intersection = np.logical_and(gt, pred).sum()

    return (2 * intersection) / (gt_sum + pred_sum)


# -------------------------
# Evaluation
# -------------------------

records = []

image_paths = sorted(
    list(VAL_IMAGES.glob("*.jpg")) +
    list(VAL_IMAGES.glob("*.jpeg")) +
    list(VAL_IMAGES.glob("*.png"))
)

print(f"Validation images: {len(image_paths)}")

for i, image_path in enumerate(image_paths):

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    h, w = image.shape[:2]

    label_path = VAL_LABELS / f"{image_path.stem}.txt"

    gt_mask = load_yolo_mask(label_path, h, w)

    # Prediction
    results = model.predict(
        source=str(image_path),
        imgsz=448,
        conf=0.25,
        iou=0.7,
        device=0,
        verbose=False
    )

    result = results[0]

    pred_mask = np.zeros((h, w), dtype=np.uint8)

    if result.masks is not None:

        masks = result.masks.data

        for mask in masks:
            mask_np = mask.cpu().numpy()

            mask_np = cv2.resize(
                mask_np,
                (w, h),
                interpolation=cv2.INTER_NEAREST
            )

            pred_mask = np.maximum(
                pred_mask,
                (mask_np > 0.5).astype(np.uint8)
            )

    gt_pixels = int(gt_mask.sum())
    pred_pixels = int(pred_mask.sum())

    iou = calculate_iou(gt_mask, pred_mask)
    dice = calculate_dice(gt_mask, pred_mask)

    gt_has_crack = gt_pixels > 0
    pred_has_crack = pred_pixels > 0

    if gt_has_crack and not pred_has_crack:
        error_type = "missed_crack"

    elif not gt_has_crack and pred_has_crack:
        error_type = "false_positive"

    elif gt_has_crack and pred_has_crack:
        error_type = "detected"

    else:
        error_type = "correct_empty"

    gt_area_pct = (gt_pixels / (h * w)) * 100

    records.append({
        "image": image_path.name,
        "gt_pixels": gt_pixels,
        "gt_area_pct": gt_area_pct,
        "pred_pixels": pred_pixels,
        "iou": iou,
        "dice": dice,
        "error_type": error_type
    })

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(image_paths)}")


# -------------------------
# DataFrame
# -------------------------

df = pd.DataFrame(records)

print("\n==============================")
print("EXP3 CUSTOM VALIDATION RESULTS")
print("==============================")

print(f"Validation images: {len(df)}")

# Overall
print("\n--- Overall ---")
print(f"Mean IoU:    {df['iou'].mean():.4f}")
print(f"Median IoU:  {df['iou'].median():.4f}")
print(f"Mean Dice:   {df['dice'].mean():.4f}")
print(f"Median Dice: {df['dice'].median():.4f}")


# Crack-only
crack_df = df[df["gt_pixels"] > 0].copy()

print("\n--- Crack-only ---")
print(f"Crack images: {len(crack_df)}")
print(f"Mean IoU:     {crack_df['iou'].mean():.4f}")
print(f"Median IoU:   {crack_df['iou'].median():.4f}")
print(f"Mean Dice:    {crack_df['dice'].mean():.4f}")
print(f"Median Dice:  {crack_df['dice'].median():.4f}")


# Errors
missed = (df["error_type"] == "missed_crack").sum()
false_positive = (df["error_type"] == "false_positive").sum()
correct_empty = (df["error_type"] == "correct_empty").sum()
detected = (df["error_type"] == "detected").sum()

print("\n--- Error Analysis ---")
print(f"Detected cracks:  {detected}")
print(f"Missed cracks:    {missed}")
print(f"Miss rate:        {missed / len(crack_df) * 100:.2f}%")
print(f"False positives:  {false_positive}")
print(f"Correct empty:    {correct_empty}")


# IoU distribution
print("\n--- Crack IoU Distribution ---")

print(
    f"IoU >= 0.75:       "
    f"{(crack_df['iou'] >= 0.75).sum()}"
)

print(
    f"0.50 <= IoU < 0.75: "
    f"{((crack_df['iou'] >= 0.50) & (crack_df['iou'] < 0.75)).sum()}"
)

print(
    f"IoU < 0.50:        "
    f"{(crack_df['iou'] < 0.50).sum()}"
)


# Size groups
def size_group(area):
    if area < 0.5:
        return "Very Small"
    elif area < 1.0:
        return "Small"
    elif area < 5.0:
        return "Medium"
    else:
        return "Large"


crack_df["size_group"] = crack_df["gt_area_pct"].apply(size_group)

print("\n--- Size Analysis ---")

for group in ["Very Small", "Small", "Medium", "Large"]:

    subset = crack_df[crack_df["size_group"] == group]

    if len(subset) == 0:
        continue

    poor = (subset["iou"] < 0.50).sum()
    missed_group = (subset["error_type"] == "missed_crack").sum()

    print(f"\n{group}")
    print(f"Count:       {len(subset)}")
    print(f"Mean IoU:    {subset['iou'].mean():.4f}")
    print(f"Mean Dice:   {subset['dice'].mean():.4f}")
    print(f"Poor IoU:    {poor}/{len(subset)}")
    print(f"Missed:      {missed_group}/{len(subset)}")


# Save evaluation CSV
OUTPUT_CSV = "/content/runs/exp3_custom_validation.csv"
df.to_csv(OUTPUT_CSV, index=False)

print("\n==============================")
print(f"Saved results to:")
print(OUTPUT_CSV)
print("==============================")

Validation images: 697
Processed 50/697
Processed 100/697
Processed 150/697
Processed 200/697
Processed 250/697
Processed 300/697
Processed 350/697
Processed 400/697
Processed 450/697
Processed 500/697
Processed 550/697
Processed 600/697
Processed 650/697

EXP3 CUSTOM VALIDATION RESULTS
Validation images: 697

--- Overall ---
Mean IoU:    0.7125
Median IoU:  0.7233
Mean Dice:   0.7970
Median Dice: 0.8394

--- Crack-only ---
Crack images: 458
Mean IoU:     0.5691
Median IoU:   0.6031
Mean Dice:    0.6976
Median Dice:  0.7524

--- Error Analysis ---
Detected cracks:  435
Missed cracks:    23
Miss rate:        5.02%
False positives:  3
Correct empty:    236

--- Crack IoU Distribution ---
IoU >= 0.75:       83
0.50 <= IoU < 0.75: 249
IoU < 0.50:        126

--- Size Analysis ---

Very Small
Count:       8
Mean IoU:    0.1759
Mean Dice:   0.2318
Poor IoU:    6/8
Missed:      4/8

Small
Count:       28
Mean IoU:    0.4070
Mean Dice:   0.5361
Poor IoU:    16/28
Missed:      4/28

Medium
Coun

In [45]:
# ============================================================
# SAVE COMPLETE CRACK SEGMENTATION PROJECT TO GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil
import os

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Define paths
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/crack-segmentation-project"
)

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("📁 Saving project to:")
print(DRIVE_ROOT)


# ------------------------------------------------------------
# 3. Helper function
# ------------------------------------------------------------

def copy_folder(source, destination):
    source = Path(source)
    destination = Path(destination)

    if not source.exists():
        print(f"⚠️ NOT FOUND: {source}")
        return

    shutil.copytree(
        source,
        destination,
        dirs_exist_ok=True
    )

    print(f"✅ Saved: {source}")
    print(f"   → {destination}")


# ------------------------------------------------------------
# 4. Save YOLO datasets
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASETS")
print("=" * 60)

# Original/current YOLO dataset
copy_folder(
    "/content/yolo_crack",
    DRIVE_ROOT / "datasets" / "yolo_crack"
)

# Experiment 3 dataset
copy_folder(
    "/content/yolo_crack_exp3",
    DRIVE_ROOT / "datasets" / "yolo_crack_exp3"
)


# ------------------------------------------------------------
# 5. Save training runs
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAINING RUNS")
print("=" * 60)

RUNS = {
    "baseline": "/content/runs/crack_seg_baseline",
    "exp1_aug": "/content/runs/crack_seg_exp1_aug",
    "exp2_no_mosaic": "/content/runs/crack_seg_exp2_no_mosaic",
    "exp3_crop": "/content/runs/crack_seg_exp3_crop",
}

for name, source in RUNS.items():

    destination = DRIVE_ROOT / "experiments" / name

    copy_folder(
        source,
        destination
    )


# ------------------------------------------------------------
# 6. Save custom evaluation results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CUSTOM EVALUATION RESULTS")
print("=" * 60)

EVAL_FILES = [
    "/content/runs/exp3_custom_validation.csv",
]

for file_path in EVAL_FILES:

    source = Path(file_path)

    if source.exists():

        destination = (
            DRIVE_ROOT
            / "evaluation"
            / source.name
        )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination
        )

        print(f"✅ Saved: {source}")
        print(f"   → {destination}")

    else:
        print(f"⚠️ NOT FOUND: {source}")


# ------------------------------------------------------------
# 7. Save validation predictions
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION PREDICTIONS")
print("=" * 60)

copy_folder(
    "/content/runs/val_predictions",
    DRIVE_ROOT / "evaluation" / "baseline_val_predictions"
)


# ------------------------------------------------------------
# 8. Save a project summary
# ------------------------------------------------------------

summary_text = """
INTELLIGENT INFRASTRUCTURE INSPECTION SYSTEM
=============================================

Project:
Crack Segmentation using YOLO11s-seg

Official Environment:
- Ultralytics: 8.4.155
- Python: 3.13.15
- PyTorch: 2.11.0+cu128
- GPU: Tesla T4
- Image Size: 448
- Epochs: 50
- Batch Size: 16

DATASET
=======
Original dataset:
- Total images: 11298
- Total masks: 11298

YOLO dataset:
- Train: 2796
- Validation: 697
- Test: 1695

Experiment 3 training dataset:
- Train: 4593
- Validation: 697
- Test: 1695

IMPORTANT:
- Test set remained locked and untouched during experiments.
- Validation set was kept unchanged for fair comparison.


OFFICIAL EXPERIMENT COMPARISON
==============================

Baseline:
- Crack Mean IoU: 0.5588
- Crack Mean Dice: 0.6859
- Missed Cracks: 31
- Miss Rate: 6.77%
- False Positives: 0
- Mask mAP50: 0.319
- Mask mAP50-95: 0.0969

Experiment 1 - Augmentation:
- Crack Mean IoU: 0.5677
- Crack Mean Dice: 0.6992
- Missed Cracks: 18
- Miss Rate: 3.93%
- False Positives: 1
- Mask mAP50: 0.327
- Mask mAP50-95: 0.0986

Experiment 2 - No Mosaic:
- Crack Mean IoU: 0.5507
- Crack Mean Dice: 0.6782
- Missed Cracks: 34
- Miss Rate: 7.42%
- False Positives: 0
- Mask mAP50: 0.310
- Mask mAP50-95: 0.0937

Experiment 3 - Crack-Focused Cropping:
- Crack Mean IoU: 0.5691
- Crack Mean Dice: 0.6976
- Missed Cracks: 23
- Miss Rate: 5.02%
- False Positives: 3
- Mask mAP50: 0.323
- Mask mAP50-95: 0.101


EXP3 SIZE ANALYSIS
==================

Very Small:
- Count: 8
- Mean IoU: 0.1759
- Mean Dice: 0.2318
- Poor IoU: 6/8
- Missed: 4/8

Small:
- Count: 28
- Mean IoU: 0.4070
- Mean Dice: 0.5361
- Poor IoU: 16/28
- Missed: 4/28

Medium:
- Count: 215
- Mean IoU: 0.5280
- Mean Dice: 0.6658
- Poor IoU: 74/215
- Missed: 11/215

Large:
- Count: 207
- Mean IoU: 0.6488
- Mean Dice: 0.7705
- Poor IoU: 30/207
- Missed: 4/207
"""


summary_path = DRIVE_ROOT / "PROJECT_SUMMARY.txt"

summary_path.write_text(
    summary_text.strip(),
    encoding="utf-8"
)

print(f"\n✅ Saved project summary:")
print(summary_path)


# ------------------------------------------------------------
# 9. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL DRIVE CHECK")
print("=" * 60)

for path in sorted(DRIVE_ROOT.rglob("*")):

    if path.is_file():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path.relative_to(DRIVE_ROOT)}"
            f"   [{size_mb:.2f} MB]"
        )

print("\n" + "=" * 60)
print("🎉 PROJECT BACKUP COMPLETED")
print("=" * 60)

print(f"\n📁 Google Drive location:")
print(DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Saving project to:
/content/drive/MyDrive/crack-segmentation-project

DATASETS
✅ Saved: /content/yolo_crack
   → /content/drive/MyDrive/crack-segmentation-project/datasets/yolo_crack
✅ Saved: /content/yolo_crack_exp3
   → /content/drive/MyDrive/crack-segmentation-project/datasets/yolo_crack_exp3

TRAINING RUNS
✅ Saved: /content/runs/crack_seg_baseline
   → /content/drive/MyDrive/crack-segmentation-project/experiments/baseline
✅ Saved: /content/runs/crack_seg_exp1_aug
   → /content/drive/MyDrive/crack-segmentation-project/experiments/exp1_aug
✅ Saved: /content/runs/crack_seg_exp2_no_mosaic
   → /content/drive/MyDrive/crack-segmentation-project/experiments/exp2_no_mosaic
✅ Saved: /content/runs/crack_seg_exp3_crop
   → /content/drive/MyDrive/crack-segmentation-project/experiments/exp3_crop

CUSTOM EVALUATION RESULTS
✅ Saved: /content/runs/exp3_custom_validatio

PermissionError: [Errno 1] Operation not permitted: '/content/drive/MyDrive/crack-segmentation-project/datasets/yolo_crack/images/test'

In [46]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/crack-segmentation-project"
)

print("🔍 Checking important project files...\n")

important_files = [
    "PROJECT_SUMMARY.txt",

    "datasets/yolo_crack/data.yaml",
    "datasets/yolo_crack_exp3/data.yaml",

    "experiments/baseline/weights/best.pt",
    "experiments/baseline/weights/last.pt",

    "experiments/exp1_aug/weights/best.pt",
    "experiments/exp1_aug/weights/last.pt",

    "experiments/exp2_no_mosaic/weights/best.pt",
    "experiments/exp2_no_mosaic/weights/last.pt",

    "experiments/exp3_crop/weights/best.pt",
    "experiments/exp3_crop/weights/last.pt",

    "evaluation/exp3_custom_validation.csv",
]

all_ok = True

for relative_path in important_files:

    path = DRIVE_ROOT / relative_path

    try:
        exists = path.exists()
    except PermissionError:
        exists = False

    if exists:
        print(f"✅ {relative_path}")
    else:
        print(f"❌ MISSING: {relative_path}")
        all_ok = False


print("\n" + "=" * 60)

if all_ok:
    print("🎉 ALL IMPORTANT PROJECT FILES ARE SAFELY ON GOOGLE DRIVE")
else:
    print("⚠️ Some important files are missing. We need to check them.")

print("=" * 60)

🔍 Checking important project files...

✅ PROJECT_SUMMARY.txt
✅ datasets/yolo_crack/data.yaml
✅ datasets/yolo_crack_exp3/data.yaml
✅ experiments/baseline/weights/best.pt
✅ experiments/baseline/weights/last.pt
✅ experiments/exp1_aug/weights/best.pt
✅ experiments/exp1_aug/weights/last.pt
✅ experiments/exp2_no_mosaic/weights/best.pt
✅ experiments/exp2_no_mosaic/weights/last.pt
✅ experiments/exp3_crop/weights/best.pt
✅ experiments/exp3_crop/weights/last.pt
✅ evaluation/exp3_custom_validation.csv

🎉 ALL IMPORTANT PROJECT FILES ARE SAFELY ON GOOGLE DRIVE


In [47]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Paths
# ============================================================

MODEL_PATH = "/content/runs/crack_seg_exp1_aug/weights/best.pt"
VAL_IMAGES = Path("/content/yolo_crack/images/val")
VAL_LABELS = Path("/content/yolo_crack/labels/val")

# Confidence thresholds to test
CONF_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

# Keep the same IoU threshold used in our previous evaluation
IOU_THRESHOLD = 0.70

IMG_SIZE = 448
DEVICE = 0


# ============================================================
# 2. Load model
# ============================================================

model = YOLO(MODEL_PATH)

print("Model loaded:")
print(MODEL_PATH)
print()


# ============================================================
# 3. Helper functions
# ============================================================

def load_yolo_mask(label_path, h, w):
    """
    Convert YOLO segmentation labels into a binary mask.
    """
    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            list(map(float, parts[1:])),
            dtype=np.float32
        ).reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


def calculate_iou(gt, pred):
    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    if union == 0:
        return 1.0

    return intersection / union


def calculate_dice(gt, pred):
    gt_pixels = gt.sum()
    pred_pixels = pred.sum()

    if gt_pixels + pred_pixels == 0:
        return 1.0

    intersection = np.logical_and(gt, pred).sum()

    return 2 * intersection / (gt_pixels + pred_pixels)


# ============================================================
# 4. Collect validation images
# ============================================================

image_paths = sorted(
    list(VAL_IMAGES.glob("*.jpg")) +
    list(VAL_IMAGES.glob("*.jpeg")) +
    list(VAL_IMAGES.glob("*.png"))
)

print(f"Validation images: {len(image_paths)}")
print()


# ============================================================
# 5. Run threshold experiments
# ============================================================

all_results = []

for conf in CONF_THRESHOLDS:

    print("=" * 70)
    print(f"Testing confidence threshold: {conf}")
    print("=" * 70)

    records = []

    start_time = time.time()

    for i, image_path in enumerate(image_paths):

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        h, w = image.shape[:2]

        # Ground-truth mask
        label_path = VAL_LABELS / f"{image_path.stem}.txt"
        gt = load_yolo_mask(label_path, h, w)

        # Prediction
        result = model.predict(
            source=str(image_path),
            imgsz=IMG_SIZE,
            conf=conf,
            iou=IOU_THRESHOLD,
            device=DEVICE,
            verbose=False
        )[0]

        # Build combined prediction mask
        pred = np.zeros((h, w), dtype=np.uint8)

        if result.masks is not None:

            for mask in result.masks.data:

                mask = mask.cpu().numpy()

                mask = cv2.resize(
                    mask,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

                pred = np.maximum(
                    pred,
                    (mask > 0.5).astype(np.uint8)
                )

        # Pixel counts
        gt_pixels = int(gt.sum())
        pred_pixels = int(pred.sum())

        # Error type
        if gt_pixels > 0 and pred_pixels == 0:
            error_type = "missed_crack"

        elif gt_pixels == 0 and pred_pixels > 0:
            error_type = "false_positive"

        elif gt_pixels > 0 and pred_pixels > 0:
            error_type = "detected"

        else:
            error_type = "correct_empty"

        records.append({
            "image": image_path.name,
            "gt_pixels": gt_pixels,
            "pred_pixels": pred_pixels,
            "iou": calculate_iou(gt, pred),
            "dice": calculate_dice(gt, pred),
            "error_type": error_type
        })

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(image_paths)}")

    # --------------------------------------------------------
    # DataFrame
    # --------------------------------------------------------

    df = pd.DataFrame(records)

    crack_df = df[df["gt_pixels"] > 0]

    missed = int((df["error_type"] == "missed_crack").sum())
    false_positives = int(
        (df["error_type"] == "false_positive").sum()
    )
    detected = int(
        (df["error_type"] == "detected").sum()
    )
    correct_empty = int(
        (df["error_type"] == "correct_empty").sum()
    )

    crack_count = len(crack_df)

    miss_rate = (
        missed / crack_count * 100
        if crack_count > 0
        else 0
    )

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    result_summary = {
        "confidence": conf,
        "crack_mean_iou": crack_df["iou"].mean(),
        "crack_median_iou": crack_df["iou"].median(),
        "crack_mean_dice": crack_df["dice"].mean(),
        "crack_median_dice": crack_df["dice"].median(),
        "detected_cracks": detected,
        "missed_cracks": missed,
        "miss_rate_%": miss_rate,
        "false_positives": false_positives,
        "correct_empty": correct_empty,
        "runtime_min": elapsed / 60
    }

    all_results.append(result_summary)

    print()
    print(f"Confidence:       {conf}")
    print(f"Crack Mean IoU:   {crack_df['iou'].mean():.4f}")
    print(f"Crack Mean Dice:  {crack_df['dice'].mean():.4f}")
    print(f"Detected cracks:  {detected}")
    print(f"Missed cracks:    {missed}")
    print(f"Miss rate:        {miss_rate:.2f}%")
    print(f"False positives:  {false_positives}")
    print(f"Correct empty:    {correct_empty}")
    print(f"Runtime:          {elapsed / 60:.2f} min")


# ============================================================
# 6. Final comparison table
# ============================================================

threshold_results = pd.DataFrame(all_results)

print()
print("=" * 90)
print("THRESHOLD TUNING RESULTS")
print("=" * 90)

display(
    threshold_results.style.format({
        "confidence": "{:.2f}",
        "crack_mean_iou": "{:.4f}",
        "crack_median_iou": "{:.4f}",
        "crack_mean_dice": "{:.4f}",
        "crack_median_dice": "{:.4f}",
        "miss_rate_%": "{:.2f}",
        "runtime_min": "{:.2f}"
    })
)


# ============================================================
# 7. Save results
# ============================================================

output_path = "/content/runs/threshold_tuning_exp1.csv"

threshold_results.to_csv(
    output_path,
    index=False
)

print()
print(f"Saved results to:")
print(output_path)

Model loaded:
/content/runs/crack_seg_exp1_aug/weights/best.pt

Validation images: 697

Testing confidence threshold: 0.1
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence:       0.1
Crack Mean IoU:   0.5893
Crack Mean Dice:  0.7257
Detected cracks:  453
Missed cracks:    5
Miss rate:        1.09%
False positives:  5
Correct empty:    234
Runtime:          0.29 min
Testing confidence threshold: 0.15
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence:       0.15
Crack Mean IoU:   0.5850
Crack Mean Dice:  0.7206
Detected cracks:  451
Missed cracks:    7
Miss rate:        1.53%
False positives:  3
Correct empty:    236
Runtime:          0.23 min
Testing confidence threshold: 0.2
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence:       0.2
Crack Mean IoU:   0.5774
Crack Mean Dice:  0.

,confidence,crack_mean_iou,crack_median_iou,crack_mean_dice,crack_median_dice,detected_cracks,missed_cracks,miss_rate_%,false_positives,correct_empty,runtime_min
0,0.10,0.5893,0.6056,0.7257,0.7543,453,5,1.09,5,234,0.29
1,0.15,0.5850,0.6048,0.7206,0.7537,451,7,1.53,3,236,0.23
2,0.20,0.5774,0.6028,0.7115,0.7522,448,10,2.18,1,238,0.23
3,0.25,0.5677,0.6017,0.6992,0.7513,440,18,3.93,1,238,0.23
4,0.30,0.5544,0.6017,0.6809,0.7513,428,30,6.55,0,239,0.23
5,0.40,0.5006,0.5777,0.6123,0.7324,381,77,16.81,0,239,0.23
6,0.50,0.4328,0.5451,0.5244,0.7056,318,140,30.57,0,239,0.23



Saved results to:
/content/runs/threshold_tuning_exp1.csv


In [48]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Paths
# ============================================================

MODEL_PATH = "/content/runs/crack_seg_exp1_aug/weights/best.pt"
VAL_IMAGES = Path("/content/yolo_crack/images/val")
VAL_LABELS = Path("/content/yolo_crack/labels/val")

# Fine-grained confidence thresholds
CONF_THRESHOLDS = [0.05, 0.07, 0.10, 0.12, 0.15, 0.18, 0.20]

# Keep exactly the same IoU threshold used previously
IOU_THRESHOLD = 0.70

IMG_SIZE = 448
DEVICE = 0


# ============================================================
# 2. Load model
# ============================================================

model = YOLO(MODEL_PATH)

print("Model loaded:")
print(MODEL_PATH)
print()


# ============================================================
# 3. Helper functions
# ============================================================

def load_yolo_mask(label_path, h, w):
    """
    Convert YOLO segmentation labels into a binary mask.
    """
    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            list(map(float, parts[1:])),
            dtype=np.float32
        ).reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


def calculate_iou(gt, pred):
    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    if union == 0:
        return 1.0

    return intersection / union


def calculate_dice(gt, pred):
    gt_pixels = gt.sum()
    pred_pixels = pred.sum()

    if gt_pixels + pred_pixels == 0:
        return 1.0

    intersection = np.logical_and(gt, pred).sum()

    return 2 * intersection / (gt_pixels + pred_pixels)


# ============================================================
# 4. Validation images
# ============================================================

image_paths = sorted(
    list(VAL_IMAGES.glob("*.jpg")) +
    list(VAL_IMAGES.glob("*.jpeg")) +
    list(VAL_IMAGES.glob("*.png"))
)

print(f"Validation images: {len(image_paths)}")
print(f"Thresholds: {CONF_THRESHOLDS}")
print()


# ============================================================
# 5. Fine-grained threshold evaluation
# ============================================================

all_results = []

for conf in CONF_THRESHOLDS:

    print("=" * 70)
    print(f"Testing confidence threshold: {conf}")
    print("=" * 70)

    records = []

    start_time = time.time()

    for i, image_path in enumerate(image_paths):

        image = cv2.imread(str(image_path))

        if image is None:
            continue

        h, w = image.shape[:2]

        # Ground-truth mask
        label_path = VAL_LABELS / f"{image_path.stem}.txt"
        gt = load_yolo_mask(label_path, h, w)

        # Prediction
        result = model.predict(
            source=str(image_path),
            imgsz=IMG_SIZE,
            conf=conf,
            iou=IOU_THRESHOLD,
            device=DEVICE,
            verbose=False
        )[0]

        # Combined prediction mask
        pred = np.zeros((h, w), dtype=np.uint8)

        if result.masks is not None:

            for mask in result.masks.data:

                mask = mask.cpu().numpy()

                mask = cv2.resize(
                    mask,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

                pred = np.maximum(
                    pred,
                    (mask > 0.5).astype(np.uint8)
                )

        # Pixel counts
        gt_pixels = int(gt.sum())
        pred_pixels = int(pred.sum())

        # Error classification
        if gt_pixels > 0 and pred_pixels == 0:
            error_type = "missed_crack"

        elif gt_pixels == 0 and pred_pixels > 0:
            error_type = "false_positive"

        elif gt_pixels > 0 and pred_pixels > 0:
            error_type = "detected"

        else:
            error_type = "correct_empty"

        records.append({
            "image": image_path.name,
            "gt_pixels": gt_pixels,
            "pred_pixels": pred_pixels,
            "iou": calculate_iou(gt, pred),
            "dice": calculate_dice(gt, pred),
            "error_type": error_type
        })

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(image_paths)}")

    # ========================================================
    # Calculate metrics
    # ========================================================

    df = pd.DataFrame(records)

    crack_df = df[df["gt_pixels"] > 0]

    missed = int(
        (df["error_type"] == "missed_crack").sum()
    )

    false_positives = int(
        (df["error_type"] == "false_positive").sum()
    )

    detected = int(
        (df["error_type"] == "detected").sum()
    )

    correct_empty = int(
        (df["error_type"] == "correct_empty").sum()
    )

    crack_count = len(crack_df)

    miss_rate = (
        missed / crack_count * 100
        if crack_count > 0
        else 0
    )

    elapsed = time.time() - start_time

    summary = {
        "confidence": conf,
        "crack_mean_iou": crack_df["iou"].mean(),
        "crack_median_iou": crack_df["iou"].median(),
        "crack_mean_dice": crack_df["dice"].mean(),
        "crack_median_dice": crack_df["dice"].median(),
        "detected_cracks": detected,
        "missed_cracks": missed,
        "miss_rate_%": miss_rate,
        "false_positives": false_positives,
        "correct_empty": correct_empty,
        "runtime_min": elapsed / 60
    }

    all_results.append(summary)

    # ========================================================
    # Print result
    # ========================================================

    print()
    print(f"Confidence:       {conf}")
    print(f"Crack Mean IoU:   {crack_df['iou'].mean():.4f}")
    print(f"Crack Mean Dice:  {crack_df['dice'].mean():.4f}")
    print(f"Detected cracks:  {detected}")
    print(f"Missed cracks:    {missed}")
    print(f"Miss rate:        {miss_rate:.2f}%")
    print(f"False positives:  {false_positives}")
    print(f"Correct empty:    {correct_empty}")
    print(f"Runtime:          {elapsed / 60:.2f} min")


# ============================================================
# 6. Comparison table
# ============================================================

threshold_results = pd.DataFrame(all_results)

print()
print("=" * 100)
print("FINE-GRAINED THRESHOLD TUNING RESULTS")
print("=" * 100)

display(
    threshold_results.style.format({
        "confidence": "{:.2f}",
        "crack_mean_iou": "{:.4f}",
        "crack_median_iou": "{:.4f}",
        "crack_mean_dice": "{:.4f}",
        "crack_median_dice": "{:.4f}",
        "miss_rate_%": "{:.2f}",
        "runtime_min": "{:.2f}"
    })
)


# ============================================================
# 7. Save results locally
# ============================================================

output_path = "/content/runs/threshold_tuning_exp1_fine.csv"

threshold_results.to_csv(
    output_path,
    index=False
)

print()
print("Saved results to:")
print(output_path)

Model loaded:
/content/runs/crack_seg_exp1_aug/weights/best.pt

Validation images: 697
Thresholds: [0.05, 0.07, 0.1, 0.12, 0.15, 0.18, 0.2]

Testing confidence threshold: 0.05
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence:       0.05
Crack Mean IoU:   0.5901
Crack Mean Dice:  0.7269
Detected cracks:  456
Missed cracks:    2
Miss rate:        0.44%
False positives:  8
Correct empty:    231
Runtime:          0.26 min
Testing confidence threshold: 0.07
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence:       0.07
Crack Mean IoU:   0.5908
Crack Mean Dice:  0.7275
Detected cracks:  454
Missed cracks:    4
Miss rate:        0.87%
False positives:  7
Correct empty:    232
Runtime:          0.25 min
Testing confidence threshold: 0.1
Processed 100/697
Processed 200/697
Processed 300/697
Processed 400/697
Processed 500/697
Processed 600/697

Confidence: 

,confidence,crack_mean_iou,crack_median_iou,crack_mean_dice,crack_median_dice,detected_cracks,missed_cracks,miss_rate_%,false_positives,correct_empty,runtime_min
0,0.05,0.5901,0.6051,0.7269,0.7540,456,2,0.44,8,231,0.26
1,0.07,0.5908,0.6056,0.7275,0.7544,454,4,0.87,7,232,0.25
2,0.10,0.5893,0.6056,0.7257,0.7543,453,5,1.09,5,234,0.24
3,0.12,0.5879,0.6048,0.7239,0.7537,452,6,1.31,4,235,0.24
4,0.15,0.5850,0.6048,0.7206,0.7537,451,7,1.53,3,236,0.23
5,0.18,0.5805,0.6045,0.7153,0.7535,449,9,1.97,1,238,0.24
6,0.20,0.5774,0.6028,0.7115,0.7522,448,10,2.18,1,238,0.24



Saved results to:
/content/runs/threshold_tuning_exp1_fine.csv


In [49]:
import matplotlib.pyplot as plt
from pathlib import Path
import cv2
import numpy as np

# ============================================================
# Configuration
# ============================================================

MODEL_PATH = "/content/runs/crack_seg_exp1_aug/weights/best.pt"

VAL_IMAGES = Path("/content/yolo_crack/images/val")
VAL_LABELS = Path("/content/yolo_crack/labels/val")

CONF_THRESHOLDS = [0.05, 0.07, 0.10, 0.12, 0.15, 0.18, 0.20]

IMG_SIZE = 448
IOU_THRESHOLD = 0.70
DEVICE = 0


# ============================================================
# Load model
# ============================================================

model = YOLO(MODEL_PATH)


# ============================================================
# Helper: load GT mask
# ============================================================

def load_yolo_mask(label_path, h, w):

    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:

        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            list(map(float, parts[1:])),
            dtype=np.float32
        ).reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


# ============================================================
# Prediction function
# ============================================================

def get_prediction(image_path, conf):

    image = cv2.imread(str(image_path))

    h, w = image.shape[:2]

    result = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=conf,
        iou=IOU_THRESHOLD,
        device=DEVICE,
        verbose=False
    )[0]

    pred = np.zeros((h, w), dtype=np.uint8)

    if result.masks is not None:

        for mask in result.masks.data:

            mask = mask.cpu().numpy()

            mask = cv2.resize(
                mask,
                (w, h),
                interpolation=cv2.INTER_NEAREST
            )

            pred = np.maximum(
                pred,
                (mask > 0.5).astype(np.uint8)
            )

    return image, pred


# ============================================================
# Find representative validation images
# ============================================================

image_paths = sorted(
    list(VAL_IMAGES.glob("*.jpg")) +
    list(VAL_IMAGES.glob("*.jpeg")) +
    list(VAL_IMAGES.glob("*.png"))
)

# ------------------------------------------------------------
# Pick a few representative images automatically
# ------------------------------------------------------------

candidates = []

for image_path in image_paths:

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    h, w = image.shape[:2]

    gt = load_yolo_mask(
        VAL_LABELS / f"{image_path.stem}.txt",
        h,
        w
    )

    gt_pixels = int(gt.sum())

    if gt_pixels == 0:
        continue

    area_pct = gt_pixels / (h * w) * 100

    candidates.append({
        "path": image_path,
        "area_pct": area_pct,
        "gt_pixels": gt_pixels
    })


# Sort by crack size
candidates = sorted(
    candidates,
    key=lambda x: x["area_pct"]
)


# Select:
# - very small
# - small
# - medium
# - large
indices = [
    0,
    len(candidates) // 4,
    len(candidates) // 2,
    int(len(candidates) * 0.75),
    len(candidates) - 1
]

selected = []

for idx in indices:

    item = candidates[idx]

    if item not in selected:
        selected.append(item)


# ============================================================
# Visualization
# ============================================================

for sample in selected:

    image_path = sample["path"]

    image = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    gt = load_yolo_mask(
        VAL_LABELS / f"{image_path.stem}.txt",
        h,
        w
    )

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(18, 9)
    )

    axes = axes.flatten()

    # --------------------------------------------------------
    # Ground Truth
    # --------------------------------------------------------

    axes[0].imshow(image_rgb)
    axes[0].imshow(
        gt,
        alpha=0.45
    )

    axes[0].set_title(
        f"Ground Truth\n{image_path.name}",
        fontsize=11
    )

    axes[0].axis("off")


    # --------------------------------------------------------
    # Predictions for each confidence
    # --------------------------------------------------------

    for idx, conf in enumerate(CONF_THRESHOLDS, start=1):

        _, pred = get_prediction(
            image_path,
            conf
        )

        axes[idx].imshow(image_rgb)

        axes[idx].imshow(
            pred,
            alpha=0.45
        )

        axes[idx].set_title(
            f"Confidence = {conf}",
            fontsize=11
        )

        axes[idx].axis("off")


    # Hide unused subplot
    if len(axes) > len(CONF_THRESHOLDS) + 1:

        for idx in range(
            len(CONF_THRESHOLDS) + 1,
            len(axes)
        ):
            axes[idx].axis("off")


    fig.suptitle(
        f"Confidence Threshold Comparison\n{image_path.name}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [50]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import time
import shutil
import os

# ============================================================
# FINAL TEST CONFIGURATION
# ============================================================

MODEL_PATH = "/content/runs/crack_seg_exp1_aug/weights/best.pt"

TEST_IMAGES = Path("/content/yolo_crack/images/test")
TEST_LABELS = Path("/content/yolo_crack/labels/test")

IMG_SIZE = 448
CONF_THRESHOLD = 0.07
IOU_THRESHOLD = 0.70
DEVICE = 0

OUTPUT_DIR = Path("/content/runs/final_test_exp1")


# ============================================================
# 1. Load FINAL model
# ============================================================

model = YOLO(MODEL_PATH)

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

print(f"Model:       {MODEL_PATH}")
print(f"Confidence:  {CONF_THRESHOLD}")
print(f"IoU:         {IOU_THRESHOLD}")
print(f"Image size:  {IMG_SIZE}")
print()


# ============================================================
# 2. Collect TEST images
# ============================================================

image_paths = sorted(
    list(TEST_IMAGES.glob("*.jpg")) +
    list(TEST_IMAGES.glob("*.jpeg")) +
    list(TEST_IMAGES.glob("*.png"))
)

print(f"Test images found: {len(image_paths)}")

if len(image_paths) != 1695:
    raise ValueError(
        f"Expected 1695 test images, but found {len(image_paths)}."
    )

print("Test set check: PASS")
print()


# ============================================================
# 3. Helper functions
# ============================================================

def load_yolo_mask(label_path, h, w):

    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:

        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            list(map(float, parts[1:])),
            dtype=np.float32
        ).reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


def calculate_iou(gt, pred):

    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    if union == 0:
        return 1.0

    return intersection / union


def calculate_dice(gt, pred):

    gt_pixels = gt.sum()
    pred_pixels = pred.sum()

    if gt_pixels + pred_pixels == 0:
        return 1.0

    intersection = np.logical_and(gt, pred).sum()

    return 2 * intersection / (gt_pixels + pred_pixels)


# ============================================================
# 4. FINAL CUSTOM TEST EVALUATION
# ============================================================

records = []

start_time = time.time()

for i, image_path in enumerate(image_paths):

    image = cv2.imread(str(image_path))

    if image is None:
        print(f"WARNING: Could not read {image_path.name}")
        continue

    h, w = image.shape[:2]

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    label_path = TEST_LABELS / f"{image_path.stem}.txt"

    gt = load_yolo_mask(
        label_path,
        h,
        w
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    result = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=DEVICE,
        verbose=False
    )[0]

    # --------------------------------------------------------
    # Combine predicted masks
    # --------------------------------------------------------

    pred = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    if result.masks is not None:

        for mask in result.masks.data:

            mask = mask.cpu().numpy()

            mask = cv2.resize(
                mask,
                (w, h),
                interpolation=cv2.INTER_NEAREST
            )

            pred = np.maximum(
                pred,
                (mask > 0.5).astype(np.uint8)
            )

    # --------------------------------------------------------
    # Pixel statistics
    # --------------------------------------------------------

    gt_pixels = int(gt.sum())
    pred_pixels = int(pred.sum())

    # --------------------------------------------------------
    # Error classification
    # --------------------------------------------------------

    if gt_pixels > 0 and pred_pixels == 0:

        error_type = "missed_crack"

    elif gt_pixels == 0 and pred_pixels > 0:

        error_type = "false_positive"

    elif gt_pixels > 0 and pred_pixels > 0:

        error_type = "detected"

    else:

        error_type = "correct_empty"

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    image_iou = calculate_iou(gt, pred)
    image_dice = calculate_dice(gt, pred)

    records.append({
        "image": image_path.name,
        "gt_pixels": gt_pixels,
        "gt_area_pct": gt_pixels / (h * w) * 100,
        "pred_pixels": pred_pixels,
        "pred_area_pct": pred_pixels / (h * w) * 100,
        "iou": image_iou,
        "dice": image_dice,
        "error_type": error_type
    })

    # Progress
    if (i + 1) % 100 == 0:
        print(
            f"Processed {i + 1}/{len(image_paths)}"
        )


elapsed = time.time() - start_time


# ============================================================
# 5. Create results DataFrame
# ============================================================

df = pd.DataFrame(records)

crack_df = df[df["gt_pixels"] > 0]

# Counts
detected = int(
    (df["error_type"] == "detected").sum()
)

missed = int(
    (df["error_type"] == "missed_crack").sum()
)

false_positives = int(
    (df["error_type"] == "false_positive").sum()
)

correct_empty = int(
    (df["error_type"] == "correct_empty").sum()
)

crack_count = len(crack_df)

miss_rate = (
    missed / crack_count * 100
    if crack_count > 0
    else 0
)


# ============================================================
# 6. FINAL TEST RESULTS
# ============================================================

print()
print()
print("=" * 80)
print("🔥 FINAL TEST RESULTS 🔥")
print("=" * 80)

print()
print(f"Test images:              {len(df)}")
print(f"Crack images:              {crack_count}")
print()

print("SEGMENTATION QUALITY")
print("-" * 50)

print(
    f"Overall Mean IoU:          {df['iou'].mean():.4f}"
)

print(
    f"Overall Median IoU:        {df['iou'].median():.4f}"
)

print(
    f"Overall Mean Dice:         {df['dice'].mean():.4f}"
)

print(
    f"Overall Median Dice:       {df['dice'].median():.4f}"
)

print()

print("CRACK-ONLY SEGMENTATION")
print("-" * 50)

print(
    f"Crack Mean IoU:            {crack_df['iou'].mean():.4f}"
)

print(
    f"Crack Median IoU:          {crack_df['iou'].median():.4f}"
)

print(
    f"Crack Mean Dice:           {crack_df['dice'].mean():.4f}"
)

print(
    f"Crack Median Dice:         {crack_df['dice'].median():.4f}"
)

print()

print("DETECTION / INSPECTION")
print("-" * 50)

print(
    f"Detected cracks:           {detected}"
)

print(
    f"Missed cracks:             {missed}"
)

print(
    f"Miss rate:                 {miss_rate:.2f}%"
)

print(
    f"False positives:           {false_positives}"
)

print(
    f"Correct empty images:     {correct_empty}"
)

print()

print(
    f"Evaluation runtime:        {elapsed / 60:.2f} minutes"
)

print()
print("=" * 80)


# ============================================================
# 7. Save custom test results
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

custom_csv = OUTPUT_DIR / "final_test_custom.csv"

df.to_csv(
    custom_csv,
    index=False
)

print()
print("Custom test results saved:")
print(custom_csv)

FINAL TEST EVALUATION
Model:       /content/runs/crack_seg_exp1_aug/weights/best.pt
Confidence:  0.07
IoU:         0.7
Image size:  448

Test images found: 1695
Test set check: PASS

Processed 100/1695
Processed 200/1695
Processed 300/1695
Processed 400/1695
Processed 500/1695
Processed 600/1695
Processed 700/1695
Processed 800/1695
Processed 900/1695
Processed 1000/1695
Processed 1100/1695
Processed 1200/1695
Processed 1300/1695
Processed 1400/1695
Processed 1500/1695
Processed 1600/1695


🔥 FINAL TEST RESULTS 🔥

Test images:              1695
Crack images:              1474

SEGMENTATION QUALITY
--------------------------------------------------
Overall Mean IoU:          0.4485
Overall Median IoU:        0.3942
Overall Mean Dice:         0.5518
Overall Median Dice:       0.5654

CRACK-ONLY SEGMENTATION
--------------------------------------------------
Crack Mean IoU:            0.3719
Crack Median IoU:          0.3388
Crack Mean Dice:           0.4907
Crack Median Dice:         0.5

In [51]:
import pandas as pd
import numpy as np

# Load final test results
test_csv = "/content/runs/final_test_exp1/final_test_custom.csv"
test_df = pd.read_csv(test_csv)

# Keep only images that actually contain cracks
test_cracks = test_df[test_df["gt_pixels"] > 0].copy()

# Classify crack size based on GT area percentage
def classify_crack_size(area_pct):
    if area_pct < 0.5:
        return "Very Small"
    elif area_pct < 1.0:
        return "Small"
    elif area_pct < 5.0:
        return "Medium"
    else:
        return "Large"

test_cracks["size_group"] = test_cracks["gt_area_pct"].apply(classify_crack_size)

# Summary
summary = (
    test_cracks
    .groupby("size_group")
    .agg(
        images=("image", "count"),
        mean_iou=("iou", "mean"),
        median_iou=("iou", "median"),
        mean_dice=("dice", "mean"),
        missed=("error_type", lambda x: (x == "missed_crack").sum()),
        detected=("error_type", lambda x: (x == "detected").sum())
    )
)

summary["miss_rate_%"] = (
    summary["missed"] / summary["images"] * 100
)

# Put groups in logical order
order = ["Very Small", "Small", "Medium", "Large"]
summary = summary.reindex(order)

print("=" * 80)
print("TEST ERROR ANALYSIS — CRACK SIZE")
print("=" * 80)
print()

print(f"Total test images: {len(test_df)}")
print(f"Crack images:      {len(test_cracks)}")
print()

print(summary.round(4))

print()
print("=" * 80)
print("CRACK SIZE DISTRIBUTION")
print("=" * 80)

distribution = (
    test_cracks["size_group"]
    .value_counts()
    .reindex(order)
)

distribution_pct = distribution / len(test_cracks) * 100

for group in order:
    count = distribution[group]
    pct = distribution_pct[group]
    print(f"{group:12s}: {count:4d} images ({pct:6.2f}%)")

TEST ERROR ANALYSIS — CRACK SIZE

Total test images: 1695
Crack images:      1474

            images  mean_iou  median_iou  mean_dice  missed  detected  \
size_group                                                              
Very Small      61    0.1422      0.0612     0.2086      10        51   
Small           82    0.3425      0.3833     0.4657       9        73   
Medium         939    0.3290      0.3129     0.4497     139       800   
Large          392    0.5163      0.5958     0.6381      14       378   

            miss_rate_%  
size_group               
Very Small      16.3934  
Small           10.9756  
Medium          14.8030  
Large            3.5714  

CRACK SIZE DISTRIBUTION
Very Small  :   61 images (  4.14%)
Small       :   82 images (  5.56%)
Medium      :  939 images ( 63.70%)
Large       :  392 images ( 26.59%)


In [52]:
import pandas as pd
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths
TEST_IMAGES = Path("/content/yolo_crack/images/test")
TEST_LABELS = Path("/content/yolo_crack/labels/test")
TEST_CSV = "/content/runs/final_test_exp1/final_test_custom.csv"

df = pd.read_csv(TEST_CSV)

# Keep only images containing cracks
cracks = df[df["gt_pixels"] > 0].copy()

# Add crack size group
def classify_crack_size(area_pct):
    if area_pct < 0.5:
        return "Very Small"
    elif area_pct < 1.0:
        return "Small"
    elif area_pct < 5.0:
        return "Medium"
    else:
        return "Large"

cracks["size_group"] = cracks["gt_area_pct"].apply(classify_crack_size)


def load_gt_mask(label_path, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)

    if not label_path.exists():
        return mask

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        parts = line.split()

        if len(parts) < 7:
            continue

        coords = np.array(
            list(map(float, parts[1:])),
            dtype=np.float32
        ).reshape(-1, 2)

        pts = np.round(
            coords * np.array([w, h], dtype=np.float32)
        ).astype(np.int32)

        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)

        cv2.fillPoly(mask, [pts], 1)

    return mask


def show_examples(data, title, n=5):
    data = data.head(n)

    fig, axes = plt.subplots(
        len(data),
        3,
        figsize=(12, 4 * len(data))
    )

    if len(data) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, row) in enumerate(data.iterrows()):

        image_path = TEST_IMAGES / row["image"]
        label_path = TEST_LABELS / (
            Path(row["image"]).stem + ".txt"
        )

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        gt = load_gt_mask(label_path, h, w)

        # Prediction mask is reconstructed approximately
        # from the stored pixel counts only for this view.
        # We will run the model in the next dedicated cell
        # for exact prediction overlays.

        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(
            f"{row['image']}\n"
            f"Size: {row['size_group']} | "
            f"GT area: {row['gt_area_pct']:.3f}%"
        )
        axes[row_idx, 0].axis("off")

        axes[row_idx, 1].imshow(gt, cmap="gray")
        axes[row_idx, 1].set_title("Ground Truth")
        axes[row_idx, 1].axis("off")

        axes[row_idx, 2].text(
            0.5,
            0.5,
            f"IoU: {row['iou']:.3f}\n"
            f"Dice: {row['dice']:.3f}\n"
            f"{row['error_type']}",
            ha="center",
            va="center",
            fontsize=14
        )
        axes[row_idx, 2].set_title("Result")
        axes[row_idx, 2].axis("off")

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------
# 1. WORST MEDIUM CRACKS
# ---------------------------------------------------------

worst_medium = (
    cracks[cracks["size_group"] == "Medium"]
    .sort_values("iou")
)

print("Worst Medium Crack Cases:")
display(
    worst_medium[
        [
            "image",
            "gt_area_pct",
            "iou",
            "dice",
            "error_type"
        ]
    ].head(10)
)

show_examples(
    worst_medium,
    "Worst Medium Cracks",
    n=5
)


# ---------------------------------------------------------
# 2. MISSED CRACKS
# ---------------------------------------------------------

missed = cracks[
    cracks["error_type"] == "missed_crack"
].sort_values("gt_area_pct")

print("\nMissed Crack Cases:")
display(
    missed[
        [
            "image",
            "gt_area_pct",
            "iou",
            "dice",
            "size_group"
        ]
    ].head(10)
)

show_examples(
    missed,
    "Missed Cracks",
    n=5
)


# ---------------------------------------------------------
# 3. BEST LARGE CRACKS
# ---------------------------------------------------------

best_large = (
    cracks[cracks["size_group"] == "Large"]
    .sort_values("iou", ascending=False)
)

print("\nBest Large Crack Cases:")
display(
    best_large[
        [
            "image",
            "gt_area_pct",
            "iou",
            "dice",
            "error_type"
        ]
    ].head(10)
)

show_examples(
    best_large,
    "Best Large Cracks",
    n=5
)

Output hidden; open in https://colab.research.google.com to view.

In [53]:
import pandas as pd
from pathlib import Path

TEST_CSV = "/content/runs/final_test_exp1/final_test_custom.csv"

df = pd.read_csv(TEST_CSV)

def identify_source(filename):
    name = filename.lower()

    if name.startswith("crack500"):
        return "CRACK500"

    elif name.startswith("deepcrack"):
        return "DeepCrack"

    elif name.startswith("gaps384"):
        return "GAPS384"

    elif name.startswith("cracktree200"):
        return "CrackTree200"

    elif name.startswith("rissbilder"):
        return "Rissbilder"

    elif name.startswith("volker"):
        return "Volker"

    else:
        return "Other / Unknown"


df["source"] = df["image"].apply(identify_source)

# Only images that actually contain cracks
cracks = df[df["gt_pixels"] > 0].copy()

source_summary = (
    cracks
    .groupby("source")
    .agg(
        images=("image", "count"),
        mean_iou=("iou", "mean"),
        median_iou=("iou", "median"),
        mean_dice=("dice", "mean"),
        missed=("error_type", lambda x: (x == "missed_crack").sum()),
        detected=("error_type", lambda x: (x == "detected").sum())
    )
)

source_summary["miss_rate_%"] = (
    source_summary["missed"] /
    source_summary["images"] * 100
)

source_summary["percentage_of_test_cracks_%"] = (
    source_summary["images"] /
    len(cracks) * 100
)

source_summary = source_summary.sort_values(
    "mean_iou",
    ascending=False
)

print("=" * 100)
print("TEST ERROR ANALYSIS — SOURCE / DATASET")
print("=" * 100)
print()

print(source_summary.round(4))

print()
print("=" * 100)
print("IMAGE COUNT BY SOURCE")
print("=" * 100)

print(
    df["source"]
    .value_counts()
    .to_string()
)

print()
print("=" * 100)
print("UNKNOWN FILES")
print("=" * 100)

unknown = df[df["source"] == "Other / Unknown"]

print(f"Unknown images: {len(unknown)}")

if len(unknown) > 0:
    print(unknown["image"].head(30).to_list())

TEST ERROR ANALYSIS — SOURCE / DATASET

                 images  mean_iou  median_iou  mean_dice  missed  detected  \
source                                                                       
CRACK500            505    0.6054      0.6338     0.7369       4       501   
DeepCrack            78    0.6000      0.6423     0.7263       2        76   
Other / Unknown      69    0.3100      0.2698     0.4344       4        65   
GAPS384              76    0.2641      0.2955     0.3728      16        60   
Volker              148    0.2494      0.2636     0.3861       9       139   
Rissbilder          567    0.2038      0.2303     0.3102     136       431   
CrackTree200         31    0.0548      0.0577     0.1029       1        30   

                 miss_rate_%  percentage_of_test_cracks_%  
source                                                     
CRACK500              0.7921                      34.2605  
DeepCrack             2.5641                       5.2917  
Other / Unknown  

In [54]:
from pathlib import Path
from collections import Counter
import pandas as pd

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

SPLITS = {
    "Train": Path("/content/yolo_crack/images/train"),
    "Validation": Path("/content/yolo_crack/images/val"),
    "Test": Path("/content/yolo_crack/images/test"),
}


# ---------------------------------------------------------
# Source identification
# ---------------------------------------------------------

def identify_source(filename):
    name = filename.lower()

    if name.startswith("crack500"):
        return "CRACK500"

    elif name.startswith("deepcrack"):
        return "DeepCrack"

    elif name.startswith("gaps384"):
        return "GAPS384"

    elif name.startswith("cracktree200"):
        return "CrackTree200"

    elif name.startswith("rissbilder"):
        return "Rissbilder"

    elif name.startswith("volker"):
        return "Volker"

    elif name.startswith("cfd_"):
        return "CFD"

    elif name.startswith("eugen_muller"):
        return "Eugen Muller"

    elif name.startswith("sylvie_chambon"):
        return "Sylvie Chambon"

    else:
        return "Other / Unknown"


# ---------------------------------------------------------
# Count sources
# ---------------------------------------------------------

rows = []

for split_name, split_path in SPLITS.items():

    image_paths = sorted(
        list(split_path.glob("*.jpg")) +
        list(split_path.glob("*.jpeg")) +
        list(split_path.glob("*.png"))
    )

    counter = Counter(
        identify_source(p.name)
        for p in image_paths
    )

    total = len(image_paths)

    for source, count in counter.items():

        rows.append({
            "split": split_name,
            "source": source,
            "images": count,
            "percentage": count / total * 100
        })


source_df = pd.DataFrame(rows)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("=" * 100)
print("SOURCE DISTRIBUTION — TRAIN / VALIDATION / TEST")
print("=" * 100)
print()

pivot_counts = source_df.pivot_table(
    index="source",
    columns="split",
    values="images",
    fill_value=0
)

pivot_pct = source_df.pivot_table(
    index="source",
    columns="split",
    values="percentage",
    fill_value=0
)

print("IMAGE COUNTS")
print("-" * 100)
print(pivot_counts.astype(int))

print()
print("PERCENTAGES WITHIN EACH SPLIT")
print("-" * 100)
print(pivot_pct.round(2))

print()
print("=" * 100)
print("TOTAL IMAGES PER SPLIT")
print("=" * 100)

for split in ["Train", "Validation", "Test"]:
    total = (source_df[source_df["split"] == split]["images"]).sum()
    print(f"{split:12s}: {total}")

SOURCE DISTRIBUTION — TRAIN / VALIDATION / TEST

IMAGE COUNTS
----------------------------------------------------------------------------------------------------
split            Test  Train  Validation
source                                  
CFD                18     78          22
CRACK500          505   1758         436
CrackTree200       31      0           0
DeepCrack          78      0           0
Eugen Muller        8      0           0
GAPS384            76      0           0
Other / Unknown   230    960         239
Rissbilder        573      0           0
Sylvie Chambon     28      0           0
Volker            148      0           0

PERCENTAGES WITHIN EACH SPLIT
----------------------------------------------------------------------------------------------------
split             Test  Train  Validation
source                                   
CFD               1.06   2.79        3.16
CRACK500         29.79  62.88       62.55
CrackTree200      1.83   0.00        0.00
Dee

In [55]:
from pathlib import Path
from collections import Counter
import pandas as pd

SPLITS = {
    "Train": Path("/content/yolo_crack/images/train"),
    "Validation": Path("/content/yolo_crack/images/val"),
    "Test": Path("/content/yolo_crack/images/test"),
}


# ---------------------------------------------------------
# Get all image filenames
# ---------------------------------------------------------

all_images = []

for split_name, split_path in SPLITS.items():

    image_paths = sorted(
        list(split_path.glob("*.jpg")) +
        list(split_path.glob("*.jpeg")) +
        list(split_path.glob("*.png"))
    )

    for p in image_paths:
        all_images.append({
            "split": split_name,
            "filename": p.name
        })


df = pd.DataFrame(all_images)


# ---------------------------------------------------------
# Show filenames that are NOT already identified
# ---------------------------------------------------------

known_prefixes = [
    "crack500",
    "deepcrack",
    "gaps384",
    "cracktree200",
    "rissbilder",
    "volker",
    "cfd_",
    "eugen_muller",
    "sylvie_chambon"
]


def is_known(filename):
    name = filename.lower()

    return any(
        name.startswith(prefix)
        for prefix in known_prefixes
    )


unknown = df[
    ~df["filename"].apply(is_known)
].copy()


# ---------------------------------------------------------
# Extract filename patterns
# ---------------------------------------------------------

def get_pattern(filename):

    name = Path(filename).stem

    # First token before underscore
    first_token = name.split("_")[0]

    return first_token


unknown["pattern"] = unknown["filename"].apply(get_pattern)


# ---------------------------------------------------------
# Pattern frequency
# ---------------------------------------------------------

pattern_counts = (
    unknown
    .groupby(["pattern", "split"])
    .size()
    .unstack(fill_value=0)
)

pattern_counts["Total"] = pattern_counts.sum(axis=1)

pattern_counts = pattern_counts.sort_values(
    "Total",
    ascending=False
)


print("=" * 100)
print("UNKNOWN / OTHER SOURCE INVESTIGATION")
print("=" * 100)
print()

print(f"Total unknown images: {len(unknown)}")
print()

print("PATTERN DISTRIBUTION")
print("-" * 100)

print(pattern_counts)


# ---------------------------------------------------------
# Show representative filenames for each pattern
# ---------------------------------------------------------

print()
print("=" * 100)
print("REPRESENTATIVE FILENAMES")
print("=" * 100)

for pattern in pattern_counts.index:

    examples = (
        unknown[
            unknown["pattern"] == pattern
        ]["filename"]
        .head(10)
        .tolist()
    )

    print()
    print(f"[{pattern}]")
    for filename in examples:
        print("  ", filename)


# ---------------------------------------------------------
# Full unknown count by split
# ---------------------------------------------------------

print()
print("=" * 100)
print("UNKNOWN COUNT BY SPLIT")
print("=" * 100)

print(
    unknown["split"]
    .value_counts()
)

UNKNOWN / OTHER SOURCE INVESTIGATION

Total unknown images: 1429

PATTERN DISTRIBUTION
----------------------------------------------------------------------------------------------------
split     Test  Train  Validation  Total
pattern                                 
noncrack   212    960         239   1411
forest      18      0           0     18

REPRESENTATIVE FILENAMES

[noncrack]
   noncrack_noncrack_concrete_wall_0_1.jpg.jpg
   noncrack_noncrack_concrete_wall_0_2.jpg.jpg
   noncrack_noncrack_concrete_wall_100_0.jpg.jpg
   noncrack_noncrack_concrete_wall_100_1.jpg.jpg
   noncrack_noncrack_concrete_wall_100_3.jpg.jpg
   noncrack_noncrack_concrete_wall_100_5.jpg.jpg
   noncrack_noncrack_concrete_wall_101_0.jpg.jpg
   noncrack_noncrack_concrete_wall_101_1.jpg.jpg
   noncrack_noncrack_concrete_wall_101_10.jpg.jpg
   noncrack_noncrack_concrete_wall_101_5.jpg.jpg

[forest]
   forest_003.jpg
   forest_006.jpg
   forest_018.jpg
   forest_022.jpg
   forest_038.jpg
   forest_048.jpg
   fo

In [57]:
from pathlib import Path

print("=" * 100)
print("CHECKING COLAB FILES")
print("=" * 100)

print("\n/content:")
for p in sorted(Path("/content").iterdir()):
    print(" ", p)

print("\nGoogle Drive:")
drive = Path("/content/drive/MyDrive/crack-segmentation-project")

if drive.exists():
    for p in sorted(drive.iterdir()):
        print(" ", p)
else:
    print("Drive project folder not found.")

CHECKING COLAB FILES

/content:
  /content/.config
  /content/drive
  /content/runs
  /content/sample_data
  /content/validation_error_analysis.csv
  /content/validation_error_analysis_exp1.csv
  /content/validation_error_analysis_exp2.csv
  /content/weights
  /content/yolo11s-seg.pt
  /content/yolo_crack
  /content/yolo_crack.zip
  /content/yolo_crack_exp3

Google Drive:
  /content/drive/MyDrive/crack-segmentation-project/PROJECT_SUMMARY.txt
  /content/drive/MyDrive/crack-segmentation-project/crack_seg_exp3_crop
  /content/drive/MyDrive/crack-segmentation-project/datasets
  /content/drive/MyDrive/crack-segmentation-project/evaluation
  /content/drive/MyDrive/crack-segmentation-project/experiments
  /content/drive/MyDrive/crack-segmentation-project/yolo_crack_exp3


In [58]:
import pandas as pd
from pathlib import Path

# Final test results
csv_path = Path("/content/runs/final_test_exp1/final_test_custom.csv")

df = pd.read_csv(csv_path)

print("=" * 110)
print("FINAL TEST — SOURCE-LEVEL ERROR ANALYSIS")
print("=" * 110)

print("Columns:")
print(df.columns.tolist())
print()
print(f"Total test images: {len(df)}")

FINAL TEST — SOURCE-LEVEL ERROR ANALYSIS
Columns:
['image', 'gt_pixels', 'gt_area_pct', 'pred_pixels', 'pred_area_pct', 'iou', 'dice', 'error_type']

Total test images: 1695


In [59]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/runs/final_test_exp1/final_test_custom.csv")
df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# 1. SOURCE CLASSIFICATION
# ---------------------------------------------------------

def classify_source(filename):
    name = Path(filename).name.lower()

    if name.startswith("crack500"):
        return "CRACK500"
    elif name.startswith("deepcrack"):
        return "DeepCrack"
    elif name.startswith("gaps384"):
        return "GAPS384"
    elif name.startswith("cracktree200"):
        return "CrackTree200"
    elif name.startswith("rissbilder"):
        return "Rissbilder"
    elif name.startswith("volker"):
        return "Volker"
    elif name.startswith("cfd_"):
        return "CFD"
    elif name.startswith("eugen_muller"):
        return "Eugen Muller"
    elif name.startswith("sylvie_chambon"):
        return "Sylvie Chambon"
    elif name.startswith("noncrack"):
        return "noncrack"
    elif name.startswith("forest"):
        return "forest"
    else:
        return "Other / Unknown"


df["source"] = df["image"].apply(classify_source)

# ---------------------------------------------------------
# 2. CRACK-ONLY SOURCE ANALYSIS
# ---------------------------------------------------------

crack_df = df[df["gt_pixels"] > 0].copy()

rows = []

for source, group in crack_df.groupby("source"):

    total = len(group)

    missed = (group["error_type"] == "FN").sum()

    detected = total - missed

    mean_iou = group["iou"].mean()
    median_iou = group["iou"].median()

    mean_dice = group["dice"].mean()
    median_dice = group["dice"].median()

    miss_rate = missed / total * 100

    rows.append({
        "Source": source,
        "Crack Images": total,
        "Mean IoU": mean_iou,
        "Median IoU": median_iou,
        "Mean Dice": mean_dice,
        "Median Dice": median_dice,
        "Missed": missed,
        "Miss Rate %": miss_rate
    })

source_results = pd.DataFrame(rows)

source_results = source_results.sort_values(
    "Crack Images",
    ascending=False
)

# ---------------------------------------------------------
# 3. SHARE OF ALL MISSED CRACKS
# ---------------------------------------------------------

total_missed = (
    crack_df["error_type"] == "FN"
).sum()

source_results["Share of All Misses %"] = (
    source_results["Missed"] /
    total_missed * 100
)

# ---------------------------------------------------------
# 4. PRINT RESULTS
# ---------------------------------------------------------

print("=" * 120)
print("FINAL TEST — SOURCE-LEVEL PERFORMANCE")
print("=" * 120)

print(
    source_results.to_string(
        index=False,
        formatters={
            "Mean IoU": "{:.4f}".format,
            "Median IoU": "{:.4f}".format,
            "Mean Dice": "{:.4f}".format,
            "Median Dice": "{:.4f}".format,
            "Miss Rate %": "{:.2f}".format,
            "Share of All Misses %": "{:.2f}".format
        }
    )
)

print()
print("=" * 120)
print("SUMMARY")
print("=" * 120)

print(f"Total crack images: {len(crack_df)}")
print(f"Total missed cracks: {total_missed}")

print()
print("Misses by source:")
print(
    source_results[
        ["Source", "Missed", "Share of All Misses %"]
    ]
    .sort_values("Missed", ascending=False)
    .to_string(
        index=False,
        formatters={
            "Share of All Misses %": "{:.2f}".format
        }
    )
)

FINAL TEST — SOURCE-LEVEL PERFORMANCE
        Source  Crack Images Mean IoU Median IoU Mean Dice Median Dice  Missed Miss Rate % Share of All Misses %
    Rissbilder           567   0.2038     0.2303    0.3102      0.3743       0        0.00                   NaN
      CRACK500           505   0.6054     0.6338    0.7369      0.7758       0        0.00                   NaN
        Volker           148   0.2494     0.2636    0.3861      0.4172       0        0.00                   NaN
     DeepCrack            78   0.6000     0.6423    0.7263      0.7822       0        0.00                   NaN
       GAPS384            76   0.2641     0.2955    0.3728      0.4562       0        0.00                   NaN
  CrackTree200            31   0.0548     0.0577    0.1029      0.1090       0        0.00                   NaN
Sylvie Chambon            25   0.1002     0.1080    0.1769      0.1950       0        0.00                   NaN
           CFD            18   0.4639     0.4480    0.6255

In [60]:
print("=" * 80)
print("ERROR TYPE VALUES")
print("=" * 80)

print(
    df["error_type"]
    .value_counts(dropna=False)
)

print()
print("Examples:")
print(
    df[["image", "gt_pixels", "pred_pixels", "iou", "error_type"]]
    .head(20)
    .to_string(index=False)
)

ERROR TYPE VALUES
error_type
detected          1302
correct_empty      212
missed_crack       172
false_positive       9
Name: count, dtype: int64

Examples:
                               image  gt_pixels  pred_pixels      iou error_type
                         CFD_001.jpg       2403         2015 0.597830   detected
                         CFD_007.jpg       1867         2188 0.548892   detected
                         CFD_011.jpg       1770         1492 0.420732   detected
                         CFD_013.jpg       6022         5367 0.600928   detected
                         CFD_014.jpg       1809         1550 0.397255   detected
                         CFD_019.jpg       4856         4487 0.571308   detected
                         CFD_037.jpg      13078         6596 0.267083   detected
                         CFD_040.jpg       3037         4363 0.422530   detected
                         CFD_047.jpg       2334         1787 0.284201   detected
                         CFD_070

In [61]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/runs/final_test_exp1/final_test_custom.csv")
df = pd.read_csv(csv_path)

# ---------------------------------------------------------
# SOURCE CLASSIFICATION
# ---------------------------------------------------------

def classify_source(filename):
    name = Path(filename).name.lower()

    if name.startswith("crack500"):
        return "CRACK500"
    elif name.startswith("deepcrack"):
        return "DeepCrack"
    elif name.startswith("gaps384"):
        return "GAPS384"
    elif name.startswith("cracktree200"):
        return "CrackTree200"
    elif name.startswith("rissbilder"):
        return "Rissbilder"
    elif name.startswith("volker"):
        return "Volker"
    elif name.startswith("cfd_"):
        return "CFD"
    elif name.startswith("eugen_muller"):
        return "Eugen Muller"
    elif name.startswith("sylvie_chambon"):
        return "Sylvie Chambon"
    elif name.startswith("noncrack"):
        return "noncrack"
    elif name.startswith("forest"):
        return "forest"
    else:
        return "Other / Unknown"


df["source"] = df["image"].apply(classify_source)

# ---------------------------------------------------------
# CRACK-ONLY SOURCE ANALYSIS
# ---------------------------------------------------------

crack_df = df[df["gt_pixels"] > 0].copy()

rows = []

for source, group in crack_df.groupby("source"):

    total = len(group)

    missed = (
        group["error_type"] == "missed_crack"
    ).sum()

    detected = (
        group["error_type"] == "detected"
    ).sum()

    rows.append({
        "Source": source,
        "Crack Images": total,
        "Mean IoU": group["iou"].mean(),
        "Median IoU": group["iou"].median(),
        "Mean Dice": group["dice"].mean(),
        "Median Dice": group["dice"].median(),
        "Detected": detected,
        "Missed": missed,
        "Miss Rate %": missed / total * 100
    })

source_results = pd.DataFrame(rows)

# ---------------------------------------------------------
# SHARE OF ALL MISSED CRACKS
# ---------------------------------------------------------

total_missed = (
    crack_df["error_type"] == "missed_crack"
).sum()

source_results["Share of All Misses %"] = (
    source_results["Missed"] / total_missed * 100
)

# ---------------------------------------------------------
# SORT BY NUMBER OF CRACK IMAGES
# ---------------------------------------------------------

source_results = source_results.sort_values(
    "Crack Images",
    ascending=False
)

# ---------------------------------------------------------
# PRINT FINAL TABLE
# ---------------------------------------------------------

print("=" * 130)
print("FINAL TEST — SOURCE-LEVEL PERFORMANCE")
print("=" * 130)

print(
    source_results.to_string(
        index=False,
        formatters={
            "Mean IoU": "{:.4f}".format,
            "Median IoU": "{:.4f}".format,
            "Mean Dice": "{:.4f}".format,
            "Median Dice": "{:.4f}".format,
            "Miss Rate %": "{:.2f}".format,
            "Share of All Misses %": "{:.2f}".format
        }
    )
)

print()
print("=" * 130)
print("MISSED CRACK CONTRIBUTION")
print("=" * 130)

print(f"Total crack images : {len(crack_df)}")
print(f"Total missed cracks: {total_missed}")
print()

print(
    source_results[
        ["Source", "Crack Images", "Missed", "Miss Rate %",
         "Share of All Misses %"]
    ]
    .sort_values("Missed", ascending=False)
    .to_string(
        index=False,
        formatters={
            "Miss Rate %": "{:.2f}".format,
            "Share of All Misses %": "{:.2f}".format
        }
    )
)

print()
print("=" * 130)
print("CONSISTENCY CHECK")
print("=" * 130)

print("CSV error counts:")
print(df["error_type"].value_counts().to_string())

print()
print(
    "Source-level crack images:",
    source_results["Crack Images"].sum()
)

print(
    "Source-level missed:",
    source_results["Missed"].sum()
)

FINAL TEST — SOURCE-LEVEL PERFORMANCE
        Source  Crack Images Mean IoU Median IoU Mean Dice Median Dice  Detected  Missed Miss Rate % Share of All Misses %
    Rissbilder           567   0.2038     0.2303    0.3102      0.3743       431     136       23.99                 79.07
      CRACK500           505   0.6054     0.6338    0.7369      0.7758       501       4        0.79                  2.33
        Volker           148   0.2494     0.2636    0.3861      0.4172       139       9        6.08                  5.23
     DeepCrack            78   0.6000     0.6423    0.7263      0.7822        76       2        2.56                  1.16
       GAPS384            76   0.2641     0.2955    0.3728      0.4562        60      16       21.05                  9.30
  CrackTree200            31   0.0548     0.0577    0.1029      0.1090        30       1        3.23                  0.58
Sylvie Chambon            25   0.1002     0.1080    0.1769      0.1950        21       4       16.00 

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# =========================================================
# PROJECT BACKUP — FINAL EVALUATION
# =========================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/crack-segmentation-project"
)

FINAL_DIR = DRIVE_ROOT / "evaluation" / "final_test"

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 100)
print("SAVING FINAL EVALUATION RESULTS")
print("=" * 100)


# ---------------------------------------------------------
# 1. FINAL TEST CUSTOM CSV
# ---------------------------------------------------------

final_test_csv = Path(
    "/content/runs/final_test_exp1/final_test_custom.csv"
)

if final_test_csv.exists():
    shutil.copy2(
        final_test_csv,
        FINAL_DIR / "final_test_custom.csv"
    )
    print("✓ Final test CSV saved")
else:
    print("⚠ Final test CSV not found")


# ---------------------------------------------------------
# 2. THRESHOLD TUNING RESULTS
# ---------------------------------------------------------

threshold_files = [
    "threshold_tuning_exp1.csv",
    "threshold_tuning_exp1_fine.csv"
]

for filename in threshold_files:

    source = Path("/content/runs") / filename

    if source.exists():
        shutil.copy2(
            source,
            FINAL_DIR / filename
        )
        print(f"✓ {filename} saved")
    else:
        print(f"⚠ {filename} not found")


# ---------------------------------------------------------
# 3. SOURCE-LEVEL ANALYSIS
# ---------------------------------------------------------

# Reload final test CSV
df = pd.read_csv(final_test_csv)


def classify_source(filename):

    name = Path(filename).name.lower()

    if name.startswith("crack500"):
        return "CRACK500"
    elif name.startswith("deepcrack"):
        return "DeepCrack"
    elif name.startswith("gaps384"):
        return "GAPS384"
    elif name.startswith("cracktree200"):
        return "CrackTree200"
    elif name.startswith("rissbilder"):
        return "Rissbilder"
    elif name.startswith("volker"):
        return "Volker"
    elif name.startswith("cfd_"):
        return "CFD"
    elif name.startswith("eugen_muller"):
        return "Eugen Muller"
    elif name.startswith("sylvie_chambon"):
        return "Sylvie Chambon"
    elif name.startswith("noncrack"):
        return "noncrack"
    elif name.startswith("forest"):
        return "forest"
    else:
        return "Other / Unknown"


df["source"] = df["image"].apply(classify_source)

crack_df = df[df["gt_pixels"] > 0].copy()

rows = []

for source, group in crack_df.groupby("source"):

    total = len(group)

    detected = (
        group["error_type"] == "detected"
    ).sum()

    missed = (
        group["error_type"] == "missed_crack"
    ).sum()

    rows.append({
        "Source": source,
        "Crack Images": total,
        "Mean IoU": group["iou"].mean(),
        "Median IoU": group["iou"].median(),
        "Mean Dice": group["dice"].mean(),
        "Median Dice": group["dice"].median(),
        "Detected": detected,
        "Missed": missed,
        "Miss Rate %": missed / total * 100
    })


source_results = pd.DataFrame(rows)

total_missed = (
    crack_df["error_type"] == "missed_crack"
).sum()

source_results["Share of All Misses %"] = (
    source_results["Missed"] /
    total_missed * 100
)

source_results = source_results.sort_values(
    "Crack Images",
    ascending=False
)

source_results.to_csv(
    FINAL_DIR / "final_test_source_analysis.csv",
    index=False
)

print("✓ Source-level analysis saved")


# ---------------------------------------------------------
# 4. SOURCE DISTRIBUTION MAP
# ---------------------------------------------------------

source_distribution = (
    df.groupby(["source"])
      .size()
      .reset_index(name="Total Test Images")
      .sort_values("Total Test Images", ascending=False)
)

source_distribution.to_csv(
    FINAL_DIR / "final_test_source_distribution.csv",
    index=False
)

print("✓ Source distribution saved")


# ---------------------------------------------------------
# 5. FINAL PROJECT METRICS SUMMARY
# ---------------------------------------------------------

summary = {
    "Model": "YOLO11s-seg",
    "Experiment": "Exp1 Small-Crack Focused Augmentation",
    "Ultralytics": "8.4.155",
    "Image Size": 448,
    "Confidence Threshold": 0.07,
    "IoU NMS Threshold": 0.70,

    "Test Images": len(df),
    "Test Crack Images": int((df["gt_pixels"] > 0).sum()),

    "Overall Mean IoU": df["iou"].mean(),
    "Overall Median IoU": df["iou"].median(),

    "Overall Mean Dice": df["dice"].mean(),
    "Overall Median Dice": df["dice"].median(),

    "Crack Mean IoU": crack_df["iou"].mean(),
    "Crack Median IoU": crack_df["iou"].median(),

    "Crack Mean Dice": crack_df["dice"].mean(),
    "Crack Median Dice": crack_df["dice"].median(),

    "Detected Cracks": int(
        (df["error_type"] == "detected").sum()
    ),

    "Missed Cracks": int(
        (df["error_type"] == "missed_crack").sum()
    ),

    "False Positives": int(
        (df["error_type"] == "false_positive").sum()
    ),

    "Correct Empty": int(
        (df["error_type"] == "correct_empty").sum()
    )
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["Metric", "Value"]
)

summary_df.to_csv(
    FINAL_DIR / "final_test_summary.csv",
    index=False
)

print("✓ Final test summary saved")


# ---------------------------------------------------------
# 6. HUMAN-READABLE TEXT SUMMARY
# ---------------------------------------------------------

summary_text = f"""
INTELLIGENT INFRASTRUCTURE INSPECTION SYSTEM
FINAL TEST EVALUATION SUMMARY

Model:
YOLO11s-seg

Experiment:
Exp1 — Small-Crack Focused Augmentation

Ultralytics:
8.4.155

Inference:
Image Size: 448
Confidence Threshold: 0.07
IoU Threshold: 0.70

FINAL TEST
----------
Test Images: {len(df)}
Crack Images: {len(crack_df)}

Overall Mean IoU: {df["iou"].mean():.4f}
Overall Median IoU: {df["iou"].median():.4f}

Overall Mean Dice: {df["dice"].mean():.4f}
Overall Median Dice: {df["dice"].median():.4f}

Crack-only Mean IoU: {crack_df["iou"].mean():.4f}
Crack-only Median IoU: {crack_df["iou"].median():.4f}

Crack-only Mean Dice: {crack_df["dice"].mean():.4f}
Crack-only Median Dice: {crack_df["dice"].median():.4f}

Detected Cracks:
{(df["error_type"] == "detected").sum()}

Missed Cracks:
{(df["error_type"] == "missed_crack").sum()}

Miss Rate:
{(df["error_type"] == "missed_crack").sum() / len(crack_df) * 100:.2f}%

False Positives:
{(df["error_type"] == "false_positive").sum()}

Correct Empty:
{(df["error_type"] == "correct_empty").sum()}


SOURCE-LEVEL FINDINGS
---------------------

Rissbilder:
Mean IoU = 0.2038
Miss Rate = 23.99%
Share of all misses = 79.07%

GAPS384:
Mean IoU = 0.2641
Miss Rate = 21.05%
Share of all misses = 9.30%

Volker:
Mean IoU = 0.2494
Miss Rate = 6.08%
Share of all misses = 5.23%

CRACK500:
Mean IoU = 0.6054
Miss Rate = 0.79%
Share of all misses = 2.33%

DeepCrack:
Mean IoU = 0.6000
Miss Rate = 2.56%
Share of all misses = 1.16%


KEY CONCLUSION
--------------

The final evaluation indicates a substantial cross-domain
generalization gap.

The model performs substantially better on source domains
represented during training, particularly CRACK500, while
performance decreases on several unseen test domains.

Rissbilder accounts for 79.07% of all missed crack images,
while Rissbilder and GAPS384 together account for 88.37%
of all missed crack images.

This indicates that source-specific visual characteristics
and domain differences are important contributors to the
observed test performance gap.

The test set was kept locked and was not used for threshold
tuning or model selection.
"""

with open(
    FINAL_DIR / "FINAL_TEST_REPORT.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(summary_text.strip())

print("✓ Human-readable final report saved")


# ---------------------------------------------------------
# 7. FINAL VERIFICATION
# ---------------------------------------------------------

print()
print("=" * 100)
print("FINAL BACKUP VERIFICATION")
print("=" * 100)

for path in sorted(FINAL_DIR.iterdir()):

    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)

        print(
            f"✓ {path.name:<45} "
            f"{size_mb:.3f} MB"
        )

print()
print(f"Backup location:")
print(FINAL_DIR)

print()
print("🎉 FINAL EVALUATION BACKUP COMPLETE")

SAVING FINAL EVALUATION RESULTS
✓ Final test CSV saved
✓ threshold_tuning_exp1.csv saved
✓ threshold_tuning_exp1_fine.csv saved
✓ Source-level analysis saved
✓ Source distribution saved
✓ Final test summary saved
✓ Human-readable final report saved

FINAL BACKUP VERIFICATION
✓ FINAL_TEST_REPORT.txt                         0.002 MB
✓ final_test_custom.csv                         0.200 MB
✓ final_test_source_analysis.csv                0.001 MB
✓ final_test_source_distribution.csv            0.000 MB
✓ final_test_summary.csv                        0.001 MB
✓ threshold_tuning_exp1.csv                     0.001 MB
✓ threshold_tuning_exp1_fine.csv                0.001 MB

Backup location:
/content/drive/MyDrive/crack-segmentation-project/evaluation/final_test

🎉 FINAL EVALUATION BACKUP COMPLETE
